# Think on the Graph: every output of the run MY11


## 0. Input
Program:
```java
for (int i = 1; i <= 2; i++) {
    for (int j = 1; j <= 3; j++) {
        System.out.println(i + " " + j);
    }
}
```
Question: In what order are the lines printed, and how many are there?

Student's answer: “Three lines. The outer loop finishes first, so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3.”
Settings: width 3, depth 3, prune llm, backend claude-cli, model claude-opus-5-5


## 1. Stage 1: the subgraph G from the code (no model)
Masked code (string and char literals and comments blanked; this is what the patterns see):
```
for (int i = 1; i <= 2; i++) {
    for (int j = 1; j <= 3; j++) {
        System.out.println(i + " " + j);
    }
}
```
Which code text tagged which concept (a generic concept fires on almost any Java line; it starts the walk only if the answer repeats its tag):
| concept | generic | channel | matched text | line |
|---|---|---|---|---|
| assignment | True | pattern | '=' | 1 |
| boolean_expression | False | pattern | 'for (int i = 1; i <= 2;' | 1 |
| comparison_operators | False | pattern | 'for (int i = 1; i <=' | 1 |
| control_flow | False | pattern | 'for (' | 1 |
| expression | True | pattern | 'i <=' | 1 |
| expression | True | pattern | '= 1;' | 1 |
| for_loop | False | pattern | 'for (' | 1 |
| increment_operators | False | pattern | '++' | 1 |
| loop_body | False | pattern | 'for (' | 1 |
| loop_control | False | signal | 'for (int i = 1; i <= 2; i++)' | 1 |
| loop_scope | False | pattern | 'for (int i' | 1 |
| loop_variable | False | pattern | 'for (int i =' | 1 |
| nested_loops | False | signal | 'for (int i = 1; i <= 2; i++) {\n    for (' | 1 |
| println | False | pattern | 'System.out.println(' | 3 |
| statement | True | pattern | ';' | 1 |
| string_concatenation | False | pattern | '+ " "' | 3 |
| string_conversion | False | pattern | '" + j' | 3 |
| string_conversion | False | pattern | 'i + "' | 3 |
| string_literal | True | pattern | '" "' | 3 |

G: 17 concept nodes (4 generic), 14 concept edges, 138 misconception / error edges, 102 misconception / error nodes, 85 level nodes, 142 level-misconception links, 3 connected piece(s).
Note: concept nodes form 3 pieces (the KG has no edge between them): assignment,boolean_expression,comparison_operators,expression,for_loop,increment_operators,loop_body,loop_control,loop_scope,loop_variable,nested_loops | control_flow,println,statement,string_literal | string_concatenation,string_conversion

Concept edges of G (the links the walk can read; `pre:` = prerequisite A -> B, `par:` = parallel):
| edge | type | rule (logic) |
|---|---|---|
| par:assignment~comparison_operators | parallel | x = y stores the value of y in x (a statement with a side effect); x == y only tests whether the two values are equal and gives a boolean. Writing = where == is meant is a compile error with int operands and a silent bug with boolean operands. |
| pre:assignment->increment_operators | prerequisite | i++ is shorthand for the assignment i = i + 1. |
| pre:comparison_operators->boolean_expression | prerequisite | A comparison such as x < y produces a boolean result; comparisons together with boolean logic provide the basis for decision-making. |
| pre:expression->comparison_operators | prerequisite | Comparison operators build expressions whose value is a boolean, so one must already know what an expression is to understand a < b. |
| pre:for_loop->loop_body | prerequisite | The statements in braces after the for header are the loop body; they run once on every pass, between the test and the increment, exactly as in a while loop. |
| pre:for_loop->loop_scope | prerequisite | A variable declared in the for header exists only from its declaration to the end of the for statement's block. |
| pre:for_loop->nested_loops | prerequisite | In nested loops the inner for runs its complete set of passes for every single pass of the outer for. |
| pre:loop_control->for_loop | prerequisite | The for header is exactly the three control slots, and the body runs after the test and before the update on each pass. |
| pre:loop_control->increment_operators | prerequisite | The loop update i = i + 1 comes first; ++ (and +=) are then introduced as shorthand for that update. |
| pre:loop_variable->loop_control | prerequisite | The init, condition and update all act on the loop variable: the init sets it, the condition tests it, the update changes it. |
| pre:statement->control_flow | prerequisite | Control flow is the order in which statements run. |
| pre:statement->println | prerequisite | System.out.println(s); is itself a statement, so its line of output appears at its own place in the sequence. |
| pre:string_concatenation->string_conversion | prerequisite | When one operand of + is a String, Java converts the other operand to a String before concatenating. |
| pre:string_literal->println | prerequisite | println("s") outputs exactly the characters between the quotes and then ends the line. |

Misconception / error edges per concept (count), and level ways per concept (count):
| concept | misconception / error edges | level ways (L1-L4 + L0 non-misconception) |
|---|---|---|
| assignment | 15 | 24 |
| boolean_expression | 8 | 19 |
| comparison_operators | 8 | 20 |
| control_flow | 11 | 15 |
| expression | 8 | 19 |
| for_loop | 21 | 24 |
| increment_operators | 5 | 16 |
| loop_body | 12 | 14 |
| loop_control | 11 | 19 |
| loop_scope | 5 | 14 |
| loop_variable | 7 | 13 |
| nested_loops | 8 | 17 |
| println | 6 | 17 |
| statement | 5 | 19 |
| string_concatenation | 3 | 16 |
| string_conversion | 1 | 13 |
| string_literal | 4 | 16 |

**Figure 1**

![Figure 1](figure01.png)
**Figure 2**

![Figure 2](figure02.png)


## 2. Clauses and start concepts (no model)
The answer split into clauses (at sentence ends, ';', and before 'because' / 'so'; quotes removed):
| clause | text |
|---|---|
| c1 | Three lines |
| c2 | The outer loop finishes first |
| c3 | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |

start: BM25 of the answer against each tagged concept: for_loop 21.85 (hits 1), loop_control 20.94 (hits 1), string_concatenation 19.33 (hits 1), string_conversion 18.38 (hits 2), increment_operators 18.11 (hits 1), nested_loops 14.66 (hits 1), control_flow 11.57 (hits 1), loop_body 9.71 (hits 1), loop_variable 9.13 (hits 1), println 6.32 (hits 1), loop_scope 5.43 (hits 1), boolean_expression 4.59 (hits 1), comparison_operators 0.89 (hits 1)

start: chosen for_loop, loop_control, string_concatenation

start concepts (3): for_loop, loop_control, string_concatenation

Start concepts: for_loop, loop_control, string_concatenation

**Figure 3**

![Figure 3](figure03.png)


## 3. The walk, depth by depth
At each depth, for each current concept: the edges offered (a concept entering as a start concept offers its misconception / error edges and its level ways, as two families; otherwise it offers its concept edges), the prune prompt and the model's reply, then the label prompt(s) and the model's reply, parsed into label / clauses / confidence. Score = prune score x confidence.

### Depth 1: current concepts 'for_loop', 'loop_control', 'string_concatenation'

#### for_loop, misconceptions: 21 edges offered
Edges offered:
| edge |
|---|
| err:for_loop->BracelessMultiStatementBody \| wrong rule: Indenting several statements under an if or for makes them all part of its body, even without braces. \| right rule: Without braces only the first statement belongs to the if/for; the rest run once, after it. |
| err:for_loop->ForCommaSeparators \| wrong rule: Writes for (int i = 0, i < n, i++). \| right rule: The three parts of a for header are separated by semicolons; commas only separate several statements inside the init or the update part. |
| err:for_loop->SemicolonAfterIfHeader \| wrong rule: if (x > 0); System.out.println("positive"); prints only for positive x. \| right rule: The ; ends the if with an empty statement, so the next line always runs. |
| err:for_loop->SemicolonAfterLoopHeader \| wrong rule: Puts a ; right after for (...) or while (...). \| right rule: A ; right after the loop header is a null (empty) loop body, so the statements written after it are not repeated by the loop. |
| err:for_loop->WrongIterationCount \| wrong rule: Updates the loop counter so that the loop runs the wrong number of times. \| right rule: The init, test and update together decide the number of passes; trace them to check the count. |
| mis:for_loop->DoubleLoopIndexCount \| wrong rule: for (double x = 0.0; x <= 1.0; x += 0.1) runs exactly 11 times. \| right rule: Using a double as a for-loop index is legal but bad practice: the number of iterations depends on an equality test between double values, which may not give the result you expect. |
| mis:for_loop->FalseLoopRepeatsNextCode \| wrong rule: A loop is an iterative if-else: while the condition is true the body repeats, and when it is false the code after the loop repeats. \| right rule: Only the body is repeated; once the condition is false the code after the loop runs once. |
| mis:for_loop->ForInitRepeats \| wrong rule: The initialization of a for loop runs again after a pass of the body. \| right rule: The initialization runs exactly once, before the first condition test. |
| mis:for_loop->ForIsConditional \| wrong rule: The body of a for statement executes at most once. \| right rule: The body of a for statement runs once for every pass on which the condition is true, which can be many times. |
| mis:for_loop->ForLoopPartSkipped \| wrong rule: A pass of a loop may leave out one of its parts (the condition test, the body, the update, or the initialization). \| right rule: Each pass of a for loop runs test, then body, then update, every time; the initialization runs once, at the start. |
| mis:for_loop->ForUpdateBeforeBody \| wrong rule: The update part of a for header (e.g. i = i - 4) runs before the body on each pass, so the first pass already sees the updated value. \| right rule: The update runs after the body, before the next condition test; the first pass uses the initial value. |
| mis:for_loop->ForWhileNotInterchangeable \| wrong rule: Some loops can only be written with for, others only with while. \| right rule: Any for loop can be rewritten as a while loop and vice versa; for just packages init, test and update. |
| mis:for_loop->IdiomOverridesCode \| wrong rule: An array loop processes the whole array from index 0 to length-1 and increments its index at the end of the body, whatever the code says. \| right rule: The loop visits exactly the indices that its initialisation, guard and update produce, in the order in which its statements are written. |
| mis:for_loop->LoopCountIsStatementCount \| wrong rule: The repeat count of a counted loop says how many of the body statements are executed. \| right rule: The repeat count is how many times the whole body runs; every pass runs all body statements. |
| mis:for_loop->LoopHeaderIsEquations \| wrong rule: The initialization, condition and update of a for header are equations that all hold at the same time. \| right rule: The three header parts are statements run at different moments: init once, then condition test, body, update, repeated. |
| mis:for_loop->LoopImpliedByOneTest \| wrong rule: Testing one case is enough; Java will repeat the test for the other values by itself. \| right rule: A statement runs once each time control reaches it; repeating a test for many values needs a loop. |
| mis:for_loop->LoopStepAlwaysOne \| wrong rule: A loop variable always goes up or down by 1, whatever the update says (so i = i - 4 is read as i--). \| right rule: The loop variable changes exactly as the update expression says (i += 2, i = i - 4, i *= 2, ...). |
| mis:for_loop->MissingBraces \| wrong rule: Omits braces around a single-statement body of if, for or while. \| right rule: Braces are optional for a single-statement body, but using them anyway avoids bugs when statements are added later. |
| mis:for_loop->ProgramEndsAfterLoop \| wrong rule: When a loop finishes the program ends, so code after the loop never runs. \| right rule: When the loop condition becomes false, control continues with the first statement after the loop. |
| mis:for_loop->ReturnDoesNotExitLoop \| wrong rule: A return inside a loop does not end the loop; it sets a value each pass (the last one is returned) or returns several values. \| right rule: When return executes, the method ends at once, so the loop stops on that pass. |
| mis:for_loop->SingleIterationInsteadOfLoop \| wrong rule: A task that must be repeated is written once, without a loop, as if one pass were enough. \| right rule: A step that must happen several times (or an unknown number of times) needs a loop such as while or for; code without a loop runs once. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:for_loop->ForLoopPartSkipped (0.50), err:for_loop->WrongIterationCount (0.30), err:for_loop->BracelessMultiStatementBody (0.20)
```
Kept by pruning: mis:for_loop->ForLoopPartSkipped (0.50), err:for_loop->WrongIterationCount (0.30), err:for_loop->BracelessMultiStatementBody (0.20)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:for_loop->ForLoopPartSkipped \| wrong rule: A pass of a loop may leave out one of its parts (the condition test, the body, the update, or the initialization). \| right rule: Each pass of a for loop runs test, then body, then update, every time; the initialization runs once, at the start. |
| err:for_loop->WrongIterationCount \| wrong rule: Updates the loop counter so that the loop runs the wrong number of times. \| right rule: The init, test and update together decide the number of passes; trace them to check the count. |
| err:for_loop->BracelessMultiStatementBody \| wrong rule: Indenting several statements under an if or for makes them all part of its body, even without braces. \| right rule: Without braces only the first statement belongs to the if/for; the rest run once, after it. |
Model's reply:
```
mis:for_loop->ForLoopPartSkipped | held | clauses: 2, 3 | confidence: 0.55
err:for_loop->WrongIterationCount | not applied | clauses: none | confidence: 0.7
err:for_loop->BracelessMultiStatementBody | not applied | clauses: none | confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:for_loop->ForLoopPartSkipped | held | 0.5 | 0.55 | 0.28 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| err:for_loop->WrongIterationCount | not applied |  |  | 0.0 |  |  |
| err:for_loop->BracelessMultiStatementBody | not applied |  |  | 0.0 |  |  |

#### for_loop, level ways: 24 edges offered
Edges offered:
| edge |
|---|
| lvl:for_loop@L0#0 \| level L0 way everyday_reading: Folk meaning: English 'for each ...' or 'for 10 times' names a whole collection or a count to be handled at once, and a maths reading treats the header as simultaneous conditions. A Java for is a while loop in compact form: one init, then test-body-update repeated, one pass at a time. |
| lvl:for_loop@L0#11 \| level L0 way inconsistent_trace: Applies the for sequencing rule (LT) unevenly: updates before the body on the first pass only, re-runs the init after a pass, or skips the test on some passes. (here mixing in: "The body of a for statement executes at most once") |
| lvl:for_loop@L1#0 \| level L1 way names_the_element: The for statement is a loop that packages an initialization, a loop-continuation condition and an increment in its header around a body. |
| lvl:for_loop@L1#1 \| level L1 way states_the_rule: Syntax: for (<initialize>; <boolean expression>; <increment>) { <statements> }, with the three parts separated by semicolons; it is, with only a few exceptions, equivalent to <initialize>; while (<boolean expression>) { <statements> <increment>; }. |
| lvl:for_loop@L2#0 \| level L2 way line_by_line_trace: Run the initializer once; then repeatedly: test the condition (stop if false), execute the body, run the update. |
| lvl:for_loop@L2#1 \| level L2 way output_prediction: A for statement has no value; the passes it makes are fixed by its header: for (int i = 1; i <= N; i++) sum += i; leaves sum = N(N+1)/2, and for (int i = 2; i <= 8; i += 2) prints 2, 4, 6, 8,. |
| lvl:for_loop@L2#2 \| level L2 way one_step_execution: One pass: the test reads the loop variable; the body applies its changes once; the update then changes the loop variable by its rule; the initializer changes state only once, before the first test. |
| lvl:for_loop@L3#0 \| level L3 way relation_to_neighbour: The for header is exactly the three loop-control slots; the body runs after the test and before the update on each pass. |
| lvl:for_loop@L3#1 \| level L3 way relation_to_neighbour: A for may be the body of another for: the inner for runs its complete set of passes for every single pass of the outer one. |
| lvl:for_loop@L3#2 \| level L3 way relation_to_neighbour: for (int i = 0; i < N; i++) gives i each valid index 0..N-1 exactly once in increasing order, the standard way to process every array element. |
| lvl:for_loop@L3#3 \| level L3 way relation_to_neighbour: A simulation runs a fixed number of trials with an outer for loop (for (int t = 0; t < T; t++)), each trial simulated inside the body. |
| lvl:for_loop@L3#4 \| level L3 way relation_to_neighbour: In a for loop, continue skips the rest of the body and transfers control directly to the increment, then the next test; this is one of the few places where the for/while equivalence fails. |
| lvl:for_loop@L3#5 \| level L3 way relation_to_neighbour: A variable declared in the for header is in scope only in the header and the body, so it cannot be used after the loop. |
| lvl:for_loop@L3#6 \| level L3 way relation_to_neighbour: A fencepost loop is a for loop that handles one item (the first or last) outside the loop so that separators come out n-1 times for n items. |
| lvl:for_loop@L3#7 \| level L3 way relation_to_neighbour: A flag search loop is a for loop that sets a boolean flag when some pass finds the target and leaves it set afterwards; the flag must be declared before the loop. |
| lvl:for_loop@L3#8 \| level L3 way purpose_of_the_whole: Plan: counted loop / counter-controlled repetition - do the body a known number of times, with the loop variable as a stepper (index, term number, trial number). |
| lvl:for_loop@L4#0 \| level L4 way predict_a_change: Rewriting a for as the equivalent while (init before, update at the end of the body) keeps behaviour, except that continue then skips the update and the loop variable stays in scope after the loop. Changing the bound or the step changes the pass count predictably (< to <= adds one pass). |
| lvl:for_loop@L4#1 \| level L4 way diagnose_an_error: rule broken: The three parts of a for header are separated by semicolons; commas only separate several statements inside the init or the update. Kind: compile-time. Trigger class: Writing for (int i = 0, i < n, i++): the compiler parses the commas as more declarators in the init and reports a syntax error. |
| lvl:for_loop@L4#2 \| level L4 way diagnose_an_error: rule broken: A ; right after while (...) or for (...) is an empty statement that is the whole loop body; the block written next is not repeated. Kind: silent logic error. Trigger class: for (...); { body } runs the body once after the loop ends; while (cond); with a cond that the empty body cannot change hangs forever (infinite loop) when cond starts true. |
| lvl:for_loop@L4#3 \| level L4 way transfer_to_a_new_program: Use for when there is an initialization, an increment and a continuation test and the loop variable is not needed after the loop; use while when the loop variable is needed later or the number of passes depends on what happens in the body. |
| lvl:for_loop@L4#4 \| level L4 way diagnose_an_error: rule broken: A ; right after if (...) is an empty statement that forms the whole then branch; the following statement or block is no longer part of the if. Kind: silent logic error. Trigger class: Any if (cond); followed by a statement or block: that statement runs for every input, whatever cond is (it compiles without error). |
| lvl:for_loop@L4#5 \| level L4 way diagnose_an_error: rule broken: The number of passes is fixed by the init, the test and the update together. Kind: silent logic error. Trigger class: Wrong start value, < vs <=, a wrong step, or an extra update of the counter in the body (or outside the loop): the loop runs too few or too many passes (off by one, or skips values). |
| lvl:for_loop@L4#6 \| level L4 way diagnose_an_error: rule broken: Without braces, the body of an if, while or for is exactly one statement; indentation does not add statements to it. Kind: silent logic error. Trigger class: A braceless header followed by two or more indented statements: only the first is controlled, the others run once, after the if/loop, on every input. |
| lvl:for_loop@L4#7 \| level L4 way diagnose_an_error: rule broken: A variable declared in the for initializer exists only inside the for statement. Kind: compile-time. Trigger class: Reading i after for (int i = 0; ...) {...}: 'cannot find symbol'. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:for_loop@L3#1 (Score: 0.6)}
{lvl:for_loop@L0#11 (Score: 0.2)}
{lvl:for_loop@L2#1 (Score: 0.2)}
```
Kept by pruning: lvl:for_loop@L3#1 (0.60), lvl:for_loop@L0#11 (0.20), lvl:for_loop@L2#1 (0.20)

Label prompt for level way lvl:for_loop@L3#1 (the model saw only the concept, the way's name 'relation to neighbour' and its text; no level):
> A for may be the body of another for: the inner for runs its complete set of passes for every single pass of the outer one.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:for_loop@L0#11 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Applies the for sequencing rule (LT) unevenly: updates before the body on the first pass only, re-runs the init after a pass, or skips the test on some passes. (here mixing in: "The body of a for statement executes at most once")

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.6
```

Label prompt for level way lvl:for_loop@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> A for statement has no value; the passes it makes are fixed by its header: for (int i = 1; i <= N; i++) sum += i; leaves sum = N(N+1)/2, and for (int i = 2; i <= 8; i += 2) prints 2, 4, 6, 8,.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.7
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:for_loop@L3#1 | not applied |  |  | 0.0 |  |  |
| lvl:for_loop@L0#11 | not applied |  |  | 0.0 |  |  |
| lvl:for_loop@L2#1 | not applied |  |  | 0.0 |  |  |

#### loop_control, misconceptions: 11 edges offered
Edges offered:
| edge |
|---|
| err:loop_control->WrongIterationCount \| wrong rule: Updates the loop counter so that the loop runs the wrong number of times. \| right rule: The init, test and update together decide the number of passes; trace them to check the count. |
| mis:loop_control->DoubleLoopIndexCount \| wrong rule: for (double x = 0.0; x <= 1.0; x += 0.1) runs exactly 11 times. \| right rule: Using a double as a for-loop index is legal but bad practice: the number of iterations depends on an equality test between double values, which may not give the result you expect. |
| mis:loop_control->ForLoopPartSkipped \| wrong rule: A pass of a loop may leave out one of its parts (the condition test, the body, the update, or the initialization). \| right rule: Each pass of a for loop runs test, then body, then update, every time; the initialization runs once, at the start. |
| mis:loop_control->ForUpdateBeforeBody \| wrong rule: The update part of a for header (e.g. i = i - 4) runs before the body on each pass, so the first pass already sees the updated value. \| right rule: The update runs after the body, before the next condition test; the first pass uses the initial value. |
| mis:loop_control->IdiomOverridesCode \| wrong rule: An array loop processes the whole array from index 0 to length-1 and increments its index at the end of the body, whatever the code says. \| right rule: The loop visits exactly the indices that its initialisation, guard and update produce, in the order in which its statements are written. |
| mis:loop_control->InfiniteLoopUnnoticed \| wrong rule: A while loop always stops on its own, even if nothing in its body can make the condition false. \| right rule: A while loop stops only when its condition is false at a check, so if no pass changes the variables in the condition, it never stops. |
| mis:loop_control->LoopHeaderIsEquations \| wrong rule: The initialization, condition and update of a for header are equations that all hold at the same time. \| right rule: The three header parts are statements run at different moments: init once, then condition test, body, update, repeated. |
| mis:loop_control->LoopStepAlwaysOne \| wrong rule: A loop variable always goes up or down by 1, whatever the update says (so i = i - 4 is read as i--). \| right rule: The loop variable changes exactly as the update expression says (i += 2, i = i - 4, i *= 2, ...). |
| mis:loop_control->LoopUpdateChangesDirection \| wrong rule: The update of the loop variable can change from pass to pass (e.g. switch direction) although the code is the same. \| right rule: The update expression is the same on every pass; the loop variable changes by the rule written in the update. |
| mis:loop_control->OffByOneBound \| wrong rule: i < n and i <= n mean the same, so the loop runs until i equals n and the last pass can use a[n]. \| right rule: for (i = 0; i < n; i++) runs n times with i going up to n - 1; with <= it runs n + 1 times and reaches i = n. |
| mis:loop_control->PreIncrementBeforeLoop \| wrong rule: With ++i in the update of for (int i = s; i < n; ++i), i is increased before the body runs, so the first pass uses s + 1. \| right rule: The update runs after each pass of the body, so with either ++i or i++ the first pass uses i = s. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:loop_control->ForLoopPartSkipped (0.50), err:loop_control->WrongIterationCount (0.30), mis:loop_control->IdiomOverridesCode (0.20)
```
Kept by pruning: mis:loop_control->ForLoopPartSkipped (0.50), err:loop_control->WrongIterationCount (0.30), mis:loop_control->IdiomOverridesCode (0.20)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:loop_control->ForLoopPartSkipped \| wrong rule: A pass of a loop may leave out one of its parts (the condition test, the body, the update, or the initialization). \| right rule: Each pass of a for loop runs test, then body, then update, every time; the initialization runs once, at the start. |
| err:loop_control->WrongIterationCount \| wrong rule: Updates the loop counter so that the loop runs the wrong number of times. \| right rule: The init, test and update together decide the number of passes; trace them to check the count. |
| mis:loop_control->IdiomOverridesCode \| wrong rule: An array loop processes the whole array from index 0 to length-1 and increments its index at the end of the body, whatever the code says. \| right rule: The loop visits exactly the indices that its initialisation, guard and update produce, in the order in which its statements are written. |
Model's reply:
```
mis:loop_control->ForLoopPartSkipped | held | clauses: 2, 3 | confidence: 0.8
err:loop_control->WrongIterationCount | not applied | clauses: none | confidence: 0.7
mis:loop_control->IdiomOverridesCode | not applied | clauses: none | confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:loop_control->ForLoopPartSkipped | held | 0.5 | 0.8 | 0.4 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| err:loop_control->WrongIterationCount | not applied |  |  | 0.0 |  |  |
| mis:loop_control->IdiomOverridesCode | not applied |  |  | 0.0 |  |  |

#### loop_control, level ways: 19 edges offered
Edges offered:
| edge |
|---|
| lvl:loop_control@L0#0 \| level L0 way everyday_reading: Folk meaning: in mathematics i = 0, i < n and i = i + 1 are equations or constraints that hold at the same time (and i = i + 1 is nonsense). In Java they are three actions run at different moments: init once, test before each pass, update after each pass. |
| lvl:loop_control@L0#3 \| level L0 way inconsistent_trace: Applies the init-test-update rule (LT) unevenly: updates before the body on some passes, skips the test or the update on a pass, or re-runs the init. (here mixing in: "++i in the for-update runs before the body each pass") |
| lvl:loop_control@L1#0 \| level L1 way names_the_element: Loop control is the init / continuation condition / increment scheme on an index variable that decides how many passes a loop makes and with which values. |
| lvl:loop_control@L1#1 \| level L1 way states_the_rule: Initialize an index variable, test a loop-continuation condition on it before each pass, and increment it as the last action of each pass; for (<initialize>; <boolean expression>; <increment>) packages the three. Choosing the loop: for when the passes are counted by an index, while when the loop runs until a condition changes, do-while when the body must run at least once; any one can be rewritten as another, so choose the clearest. |
| lvl:loop_control@L2#0 \| level L2 way line_by_line_trace: init (once) -> test -> body -> update -> test -> body -> update ... -> test false -> exit. |
| lvl:loop_control@L2#1 \| level L2 way output_prediction: The number of passes and the values seen follow from the three parts: for (int i = 0; i <= N; i++) makes N+1 passes with i = 0..N; with i < N it makes N passes; i += 2 visits every second value. |
| lvl:loop_control@L2#2 \| level L2 way one_step_execution: One update changes only the loop variable, by exactly the rule written (i++ adds 1, i = i - 4 subtracts 4); one test reads it and returns true or false without changing it. |
| lvl:loop_control@L3#0 \| level L3 way relation_to_neighbour: All three control parts act on the loop variable: the init sets it, the condition tests it, the update changes it. |
| lvl:loop_control@L3#1 \| level L3 way relation_to_neighbour: The for header is exactly the three control slots, separated by semicolons; the body runs after the test and before the update. |
| lvl:loop_control@L3#2 \| level L3 way relation_to_neighbour: The update i = i + 1 is usually written i++ (or i += k for other steps); as a whole update statement ++i and i++ are equivalent. |
| lvl:loop_control@L3#3 \| level L3 way relation_to_neighbour: A loop is infinite when the update never moves the variable toward making the condition false (e.g. condition i > 3 with update i = i + 1). |
| lvl:loop_control@L3#4 \| level L3 way relation_to_neighbour: The continuation condition fixes the number of iterations, so a sharper bound (i <= n/i instead of i < n) can make a program dramatically faster. |
| lvl:loop_control@L3#5 \| level L3 way relation_to_neighbour: The initialization and the condition of a loop fix how many times it runs; one wrong bound makes it run once too many or too few. |
| lvl:loop_control@L3#6 \| level L3 way purpose_of_the_whole: Plan: counter-controlled loop - the loop variable is a stepper that walks a predictable sequence of values (0..N-1, 1..N, every 2nd value) and fixes the number of passes in advance. |
| lvl:loop_control@L4#0 \| level L4 way predict_a_change: Changing < to <= adds one pass; starting at 1 instead of 0 removes one; doubling the step halves the passes; reversing init and bound with i-- visits the same values in reverse order (and still processes the same set only if both ends are adjusted). |
| lvl:loop_control@L4#1 \| level L4 way diagnose_an_error: rule broken: The number of passes is fixed by the init, the test and the update together. Kind: silent logic error. Trigger class: Wrong start value, < vs <=, a wrong step, or an extra update of the counter in the body (or outside the loop): the loop runs too few or too many passes (off by one, or skips values). |
| lvl:loop_control@L4#2 \| level L4 way transfer_to_a_new_program: Use explicit init/test/update control when the passes are counted or follow a fixed step; when the number of passes depends on data computed in the body, control the loop by that data (a while condition) instead. |
| lvl:loop_control@L4#3 \| level L4 way diagnose_an_error: rule broken: The update must move the loop variable toward making the condition false. Kind: silent logic error. Trigger class: An update in the wrong direction or a condition that the update cannot falsify (e.g. i > 3 with i++, or i != 10 with i += 3): infinite loop. |
| lvl:loop_control@L4#4 \| level L4 way diagnose_an_error: rule broken: Do not control a loop by an equality test on double values. Kind: silent logic error. Trigger class: for (double x = 0.0; x < 1.0; x += 0.1): after ten additions x is 0.9999999999999999, so the loop makes 11 passes instead of the intended 10. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:loop_control@L0#3 (Score: 0.5)}
{lvl:loop_control@L2#1 (Score: 0.3)}
{lvl:loop_control@L2#0 (Score: 0.2)}
```
Kept by pruning: lvl:loop_control@L0#3 (0.50), lvl:loop_control@L2#1 (0.30), lvl:loop_control@L2#0 (0.20)

Label prompt for level way lvl:loop_control@L0#3 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Applies the init-test-update rule (LT) unevenly: updates before the body on some passes, skips the test or the update on a pass, or re-runs the init. (here mixing in: "++i in the for-update runs before the body each pass")

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.8
```

Label prompt for level way lvl:loop_control@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> The number of passes and the values seen follow from the three parts: for (int i = 0; i <= N; i++) makes N+1 passes with i = 0..N; with i < N it makes N passes; i += 2 visits every second value.

Model's reply:
```
Label: shown
Clauses: 1, 3
Confidence: 0.6
```

Label prompt for level way lvl:loop_control@L2#0 (the model saw only the concept, the way's name 'line by line trace' and its text; no level):
> init (once) -> test -> body -> update -> test -> body -> update ... -> test false -> exit.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.7
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:loop_control@L0#3 | not applied |  |  | 0.0 |  |  |
| lvl:loop_control@L2#1 | shown | 0.3 | 0.6 | 0.18 | c1, c3 | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| lvl:loop_control@L2#0 | not applied |  |  | 0.0 |  |  |

#### string_concatenation, misconceptions: 3 edges offered
Edges offered:
| edge |
|---|
| mis:string_concatenation->ArithmeticPlusPrecedes \| wrong rule: In "s" + a + b, the numbers a and b are added first and their sum is joined to s. \| right rule: Both uses of + have the same precedence and are evaluated left to right, so "s" + a + b joins a and then b as text; only "s" + (a + b) adds them first. |
| mis:string_concatenation->StringPlusLeftToRight \| wrong rule: 2 + 3 + "bc" and "bc" + 2 + 3 give the same text. \| right rule: + is evaluated left to right: 2 + 3 + "bc" is "5bc" but "bc" + 2 + 3 is "bc23". |
| mis:string_concatenation->StringPlusStringifiesExpression \| wrong rule: In "s" + (a + b), the text a + b is joined to s instead of its value. \| right rule: The expression (a + b) is evaluated first, and its value is then turned into text and joined to s. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{mis:string_concatenation->ArithmeticPlusPrecedes (Score: 0.6)}
{mis:string_concatenation->StringPlusLeftToRight (Score: 0.3)}
{mis:string_concatenation->StringPlusStringifiesExpression (Score: 0.1)}
```
Kept by pruning: mis:string_concatenation->ArithmeticPlusPrecedes (0.60), mis:string_concatenation->StringPlusLeftToRight (0.30), mis:string_concatenation->StringPlusStringifiesExpression (0.10)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:string_concatenation->ArithmeticPlusPrecedes \| wrong rule: In "s" + a + b, the numbers a and b are added first and their sum is joined to s. \| right rule: Both uses of + have the same precedence and are evaluated left to right, so "s" + a + b joins a and then b as text; only "s" + (a + b) adds them first. |
| mis:string_concatenation->StringPlusLeftToRight \| wrong rule: 2 + 3 + "bc" and "bc" + 2 + 3 give the same text. \| right rule: + is evaluated left to right: 2 + 3 + "bc" is "5bc" but "bc" + 2 + 3 is "bc23". |
| mis:string_concatenation->StringPlusStringifiesExpression \| wrong rule: In "s" + (a + b), the text a + b is joined to s instead of its value. \| right rule: The expression (a + b) is evaluated first, and its value is then turned into text and joined to s. |
Model's reply:
```
mis:string_concatenation->ArithmeticPlusPrecedes | not applied | clauses: none | confidence: 0.8
mis:string_concatenation->StringPlusLeftToRight | not applied | clauses: none | confidence: 0.8
mis:string_concatenation->StringPlusStringifiesExpression | not applied | clauses: none | confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:string_concatenation->ArithmeticPlusPrecedes | not applied |  |  | 0.0 |  |  |
| mis:string_concatenation->StringPlusLeftToRight | not applied |  |  | 0.0 |  |  |
| mis:string_concatenation->StringPlusStringifiesExpression | not applied |  |  | 0.0 |  |  |

#### string_concatenation, level ways: 16 edges offered
Edges offered:
| edge |
|---|
| lvl:string_concatenation@L0#0 \| level L0 way everyday_reading: In arithmetic, + always means 'add', and 'plus' combines quantities, so novices expect "1" + 2 to be 3. In Java, + with a String operand joins text end to end, and a chain of + is processed left to right, one operator at a time. |
| lvl:string_concatenation@L0#3 \| level L0 way inconsistent_trace: Points to LT (left to right, and each + adds or concatenates according to its operands at that moment). Applied unevenly, a student keeps adding numbers after a String has appeared, or concatenates before any String appears (1 + 2 + "x" as "12x"). (here mixing in: "+ with a String turns the whole other side into a string first") |
| lvl:string_concatenation@L1#0 \| level L1 way names_the_element: String concatenation: the + operator with a String operand, which joins two values end to end into a new String. |
| lvl:string_concatenation@L1#1 \| level L1 way states_the_rule: If either operand of + is a String, the other is converted to a String, and the result is a new String made of the first operand's characters followed by the second's. |
| lvl:string_concatenation@L2#0 \| level L2 way line_by_line_trace: A chain of + is evaluated left to right, after any parenthesized parts. At each + the current operand types decide: two numbers are added, otherwise the values are concatenated. |
| lvl:string_concatenation@L2#1 \| level L2 way output_prediction: The result is a String: 2 + "bc" is "2bc", 2 + 3 + "bc" is "5bc", "bc" + 2 + 3 is "bc23", and "bc" + (2+3) is "bc5". |
| lvl:string_concatenation@L2#2 \| level L2 way one_step_execution: Concatenation creates a new String and changes no operand. A variable changes only if the result is assigned, as in ruler = ruler + i + ruler;. |
| lvl:string_concatenation@L3#0 \| level L3 way relation_to_neighbour: Concatenation always yields a String, even when one operand is a number, so its result cannot be used in arithmetic without parsing. |
| lvl:string_concatenation@L3#1 \| level L3 way relation_to_neighbour: The non-String operand is converted to its printed form (int 99 to "99", double 1.0 to "1.0", char 'a' to "a") before joining. |
| lvl:string_concatenation@L3#2 \| level L3 way relation_to_neighbour: + adds when both operands are numbers and concatenates when either operand is a String, converting the other operand to a String. It is left-associative, so 1 + 2 + "3" is "33" but "1" + 2 + 3 is "123". |
| lvl:string_concatenation@L3#3 \| level L3 way purpose_of_the_whole: Concatenation builds output lines and labels from computed values (a + " + " + b + " = " + c). Repeated s = s + piece makes s a gatherer of text. |
| lvl:string_concatenation@L4#0 \| level L4 way predict_a_change: Parenthesizing numeric parts changes the result ("sum " + (a + b) adds before joining). Moving numbers before or after the first String changes whether they are added. Concatenating into a printf format string removes the separate argument printf needs. |
| lvl:string_concatenation@L4#1 \| level L4 way transfer_to_a_new_program: Use + concatenation for simple output and labels, putting parentheses around any arithmetic inside. Use printf when numbers need formatting, and parse Strings before using them as numbers. |
| lvl:string_concatenation@L4#2 \| level L4 way diagnose_an_error: rule broken: Once a String appears in a + chain, every later + concatenates. Kind: silent logic error. Trigger class: "Sum: " + a + b with numbers a and b prints both digit strings side by side (e.g. Sum: 23). |
| lvl:string_concatenation@L4#3 \| level L4 way diagnose_an_error: rule broken: char + char is numeric addition, not concatenation. Kind: silent logic error. Trigger class: Joining two chars with + when no String is involved ('b' + 'c' prints 197). |
| lvl:string_concatenation@L4#4 \| level L4 way diagnose_an_error: rule broken: printf needs its values as separate arguments matching the format specifiers. Kind: run-time exception. Trigger class: Concatenating the value into the format (printf("inches = %d" + inch)): MissingFormatArgumentException. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:string_concatenation@L2#1 (Score: 0.45)}
{lvl:string_concatenation@L3#3 (Score: 0.3)}
{lvl:string_concatenation@L3#1 (Score: 0.25)}
```
Kept by pruning: lvl:string_concatenation@L2#1 (0.45), lvl:string_concatenation@L3#3 (0.30), lvl:string_concatenation@L3#1 (0.25)

Label prompt for level way lvl:string_concatenation@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> The result is a String: 2 + "bc" is "2bc", 2 + 3 + "bc" is "5bc", "bc" + 2 + 3 is "bc23", and "bc" + (2+3) is "bc5".

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.75
```

Label prompt for level way lvl:string_concatenation@L3#3 (the model saw only the concept, the way's name 'purpose of the whole' and its text; no level):
> Concatenation builds output lines and labels from computed values (a + " + " + b + " = " + c). Repeated s = s + piece makes s a gatherer of text.

Model's reply:
```
Label: shown
Clauses: 3
Confidence: 0.6
```

Label prompt for level way lvl:string_concatenation@L3#1 (the model saw only the concept, the way's name 'relation to neighbour' and its text; no level):
> The non-String operand is converted to its printed form (int 99 to "99", double 1.0 to "1.0", char 'a' to "a") before joining.

Model's reply:
```
Label: shown
Clauses: 3
Confidence: 0.6
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:string_concatenation@L2#1 | not applied |  |  | 0.0 |  |  |
| lvl:string_concatenation@L3#3 | shown | 0.3 | 0.6 | 0.18 | c3 | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| lvl:string_concatenation@L3#1 | shown | 0.25 | 0.6 | 0.15 | c3 | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |

**Kept on a path at this depth** (best N in tiers: held misconception / broken edge, then held error, then correct / confirmed / shown):

mis:loop_control->ForLoopPartSkipped [held, tier 0, 0.40], mis:for_loop->ForLoopPartSkipped [held, tier 0, 0.28], lvl:loop_control@L2#1 [shown, tier 2, 0.18]

**Clause coverage after this depth:**
| clause | text | covered by |
|---|---|---|
| c1 | Three lines | not covered yet |
| c2 | The outer loop finishes first | mis:for_loop->ForLoopPartSkipped |
| c3 | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | mis:for_loop->ForLoopPartSkipped |

**Uncovered clauses bring in (BM25 of the uncovered clauses against unvisited concepts):** control_flow (1.84), println (1.53)

**Graph check, not yet read (the walk may not stop while these remain):** own edges of control_flow, own edges of println, pre:for_loop->loop_body, pre:for_loop->loop_scope, pre:for_loop->nested_loops, pre:loop_control->for_loop, pre:loop_control->increment_operators, pre:loop_variable->loop_control, pre:string_concatenation->string_conversion

**Figure 4 (the graph after depth 1)**

![Figure 4](figure04.png)
### Depth 2: current concepts 'loop_control', 'for_loop', 'string_concatenation', 'control_flow', 'println'

#### loop_control, concept edges: 3 edges offered
Edges offered:
| edge |
|---|
| pre:loop_control->for_loop \| rule: The for header is exactly the three control slots, and the body runs after the test and before the update on each pass. |
| pre:loop_control->increment_operators \| rule: The loop update i = i + 1 comes first; ++ (and +=) are then introduced as shorthand for that update. |
| pre:loop_variable->loop_control \| rule: The init, condition and update all act on the loop variable: the init sets it, the condition tests it, the update changes it. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{pre:loop_control->for_loop (Score: 0.6)}
{pre:loop_variable->loop_control (Score: 0.3)}
{pre:loop_control->increment_operators (Score: 0.1)}
```
Kept by pruning: pre:loop_control->for_loop (0.60), pre:loop_variable->loop_control (0.30), pre:loop_control->increment_operators (0.10)

Label prompt for concept edge pre:loop_control->for_loop (loop_control -> for_loop, prerequisite):
> The for header is exactly the three control slots, and the body runs after the test and before the update on each pass.

Model's reply:
```
Label: broken
Clauses: 2, 3
Confidence: 0.85
```

Label prompt for concept edge pre:loop_variable->loop_control (loop_variable -> loop_control, prerequisite):
> The init, condition and update all act on the loop variable: the init sets it, the condition tests it, the update changes it.

Model's reply:
```
Label: confirmed
Clauses: 3
Confidence: 0.6
```

Label prompt for concept edge pre:loop_control->increment_operators (loop_control -> increment_operators, prerequisite):
> The loop update i = i + 1 comes first; ++ (and +=) are then introduced as shorthand for that update.

Model's reply:
```
Label: not applied
Clauses: none
Confidence: 0.7
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:loop_control->for_loop | broken | 0.6000000000000001 | 0.85 | 0.51 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| pre:loop_variable->loop_control | confirmed | 0.30000000000000004 | 0.6 | 0.18 | c3 | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| pre:loop_control->increment_operators | not applied |  |  | 0.0 |  |  |

#### for_loop, concept edges: 3 edges offered
Edges offered:
| edge |
|---|
| pre:for_loop->loop_body \| rule: The statements in braces after the for header are the loop body; they run once on every pass, between the test and the increment, exactly as in a while loop. |
| pre:for_loop->loop_scope \| rule: A variable declared in the for header exists only from its declaration to the end of the for statement's block. |
| pre:for_loop->nested_loops \| rule: In nested loops the inner for runs its complete set of passes for every single pass of the outer for. |
| pre:loop_control->for_loop \| rule: The for header is exactly the three control slots, and the body runs after the test and before the update on each pass. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{pre:for_loop->nested_loops (Score: 0.7)}
{pre:for_loop->loop_body (Score: 0.2)}
{pre:for_loop->loop_scope (Score: 0.1)}
```
Kept by pruning: pre:for_loop->nested_loops (0.70), pre:for_loop->loop_body (0.20), pre:for_loop->loop_scope (0.10)

Label prompt for concept edge pre:for_loop->nested_loops (for_loop -> nested_loops, prerequisite):
> In nested loops the inner for runs its complete set of passes for every single pass of the outer for.

Model's reply:
```
Label: broken
Clauses: 2, 3
Confidence: 0.92
```

Label prompt for concept edge pre:for_loop->loop_body (for_loop -> loop_body, prerequisite):
> The statements in braces after the for header are the loop body; they run once on every pass, between the test and the increment, exactly as in a while loop.

Model's reply:
```
Label: broken
Clauses: 2, 3
Confidence: 0.85
```

Label prompt for concept edge pre:for_loop->loop_scope (for_loop -> loop_scope, prerequisite):
> A variable declared in the for header exists only from its declaration to the end of the for statement's block.

Model's reply:
```
Label: broken
Clauses: 2, 3
Confidence: 0.6
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:for_loop->nested_loops | broken | 0.7000000000000001 | 0.92 | 0.64 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| pre:for_loop->loop_body | broken | 0.20000000000000004 | 0.85 | 0.17 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| pre:for_loop->loop_scope | broken | 0.10000000000000002 | 0.6 | 0.06 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |

#### string_concatenation, concept edges: 1 edges offered
Edges offered:
| edge |
|---|
| pre:string_concatenation->string_conversion \| rule: When one operand of + is a String, Java converts the other operand to a String before concatenating. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{pre:string_concatenation->string_conversion (Score: 1.0)}
```
Kept by pruning: pre:string_concatenation->string_conversion (1.00)

Label prompt for concept edge pre:string_concatenation->string_conversion (string_concatenation -> string_conversion, prerequisite):
> When one operand of + is a String, Java converts the other operand to a String before concatenating.

Model's reply:
```
Label: confirmed
Clauses: 3
Confidence: 0.6
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:string_concatenation->string_conversion | confirmed | 1.0 | 0.6 | 0.6 | c3 | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |

#### control_flow, misconceptions: 11 edges offered
Edges offered:
| edge |
|---|
| mis:control_flow->CodeAfterIfSkipped \| wrong rule: When the then branch runs, the code after the if is skipped. \| right rule: After the if finishes, execution continues with the next statement either way. |
| mis:control_flow->FalseConditionRestartsProgram \| wrong rule: When an if condition is false (and there is no else), control jumps back to the start of the program. \| right rule: A false condition skips the then branch and execution continues with the statement after the if statement. |
| mis:control_flow->FalseLoopRepeatsNextCode \| wrong rule: A loop is an iterative if-else: while the condition is true the body repeats, and when it is false the code after the loop repeats. \| right rule: Only the body is repeated; once the condition is false the code after the loop runs once. |
| mis:control_flow->IfConditionWatched \| wrong rule: An if statement watches its condition while the program runs and fires as soon as the condition becomes true (if = whenever). \| right rule: An if condition is tested once, when control reaches the if statement; later changes to the variables do not trigger the branch. |
| mis:control_flow->IfWaitsForCondition \| wrong rule: An IF statement earlier in the program stays active and fires later, e.g. inside a later loop, when its condition becomes true. \| right rule: An if is evaluated once when control reaches it; afterwards it is inactive unless control returns to it. |
| mis:control_flow->LoopImpliedByOneTest \| wrong rule: Testing one case is enough; Java will repeat the test for the other values by itself. \| right rule: A statement runs once each time control reaches it; repeating a test for many values needs a loop. |
| mis:control_flow->NoShortCircuit \| wrong rule: logical AND/OR always evaluate both sides (Java short-circuits them) \| right rule: In p && q, q is evaluated only when p is true; in p \|\| q, q is evaluated only when p is false. |
| mis:control_flow->ProgramEndsAfterLoop \| wrong rule: When a loop finishes the program ends, so code after the loop never runs. \| right rule: When the loop condition becomes false, control continues with the first statement after the loop. |
| mis:control_flow->ProgramLooksAhead \| wrong rule: The program knows or sees what later lines will do and acts on those goals at the current line, e.g. it stops 'at ten' because it wants to draw a box. \| right rule: Each statement executes using only the current state; later lines have no effect until control reaches them. |
| mis:control_flow->ReturnDoesNotExitLoop \| wrong rule: A return inside a loop does not end the loop; it sets a value each pass (the last one is returned) or returns several values. \| right rule: When return executes, the method ends at once, so the loop stops on that pass. |
| mis:control_flow->StatementsRunInAnyOrder \| wrong rule: Statements in a program can run in any order. \| right rule: Without control statements, statements run once each, top to bottom. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:control_flow->StatementsRunInAnyOrder (0.60), mis:control_flow->FalseLoopRepeatsNextCode (0.25), mis:control_flow->ProgramEndsAfterLoop (0.15)
```
Kept by pruning: mis:control_flow->StatementsRunInAnyOrder (0.60), mis:control_flow->FalseLoopRepeatsNextCode (0.25), mis:control_flow->ProgramEndsAfterLoop (0.15)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:control_flow->StatementsRunInAnyOrder \| wrong rule: Statements in a program can run in any order. \| right rule: Without control statements, statements run once each, top to bottom. |
| mis:control_flow->FalseLoopRepeatsNextCode \| wrong rule: A loop is an iterative if-else: while the condition is true the body repeats, and when it is false the code after the loop repeats. \| right rule: Only the body is repeated; once the condition is false the code after the loop runs once. |
| mis:control_flow->ProgramEndsAfterLoop \| wrong rule: When a loop finishes the program ends, so code after the loop never runs. \| right rule: When the loop condition becomes false, control continues with the first statement after the loop. |
Model's reply:
```
mis:control_flow->StatementsRunInAnyOrder | not applied | clauses: none | confidence: 0.6
mis:control_flow->FalseLoopRepeatsNextCode | not applied | clauses: none | confidence: 0.6
mis:control_flow->ProgramEndsAfterLoop | correct | clauses: 2, 3 | confidence: 0.55
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:control_flow->StatementsRunInAnyOrder | not applied |  |  | 0.0 |  |  |
| mis:control_flow->FalseLoopRepeatsNextCode | not applied |  |  | 0.0 |  |  |
| mis:control_flow->ProgramEndsAfterLoop | correct | 0.15 | 0.55 | 0.08 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |

#### control_flow, level ways: 15 edges offered
Edges offered:
| edge |
|---|
| lvl:control_flow@L0#0 \| level L0 way everyday_reading: Folk meaning: in everyday instructions a listener reads the whole list at once, does steps in a sensible order and infers what the writer wants. Java instead executes statements one at a time, top to bottom, each only when control reaches it, and nothing later in the program has any effect earlier. |
| lvl:control_flow@L0#3 \| level L0 way inconsistent_trace: Applies the sequencing rule (LT) unevenly: e.g. runs a later line before an earlier one, lets an if that was already passed fire again later, or ends the program when a loop or a false if ends instead of continuing with the next statement. (here mixing in: "Statements in a program can run in any order") |
| lvl:control_flow@L1#0 \| level L1 way names_the_element: Control flow is the order in which a program's statements are executed (statement sequencing); conditionals and loops are the statements that change it. |
| lvl:control_flow@L1#1 \| level L1 way states_the_rule: Without control statements every statement executes once, in the order given; a block executes its statements in order from first to last; if/if-else choose between one or two alternatives and loops repeat statements depending on a condition. |
| lvl:control_flow@L2#0 \| level L2 way line_by_line_trace: Execute the statement at the current point; when it completes normally, control passes to the next statement in the block. An if evaluates its condition once and then runs at most one branch; a loop evaluates its condition before each pass; after an if or a loop finishes, control continues with the statement after it. |
| lvl:control_flow@L2#1 \| level L2 way output_prediction: The output is the sequence of print statements that control actually reaches, in the order reached: e.g. TenHellos prints 1st, 2nd, 3rd Hello once each, then 4th to 10th Hello from the loop. |
| lvl:control_flow@L2#2 \| level L2 way one_step_execution: One step executes the single statement at the current control point using the current values, updates at most the variables it assigns, and moves the control point to the next statement (or, for an if/loop test, to the chosen branch, back to the test, or past the statement). |
| lvl:control_flow@L3#0 \| level L3 way relation_to_neighbour: Control flow is defined over statements: statements are executed for their effect, and if/while/for are themselves statements that may be used wherever a statement is allowed, so they can nest. |
| lvl:control_flow@L3#1 \| level L3 way relation_to_neighbour: An if changes the default top-to-bottom flow: its condition is tested once when control reaches it, its block runs only if true, and in both cases control then continues with the next statement. |
| lvl:control_flow@L3#2 \| level L3 way relation_to_neighbour: A flowchart draws control flow: each path through the diagram is a possible path of execution, with a decision box for each if or loop test. |
| lvl:control_flow@L3#3 \| level L3 way purpose_of_the_whole: Plan: control flow lets one program do different things for different inputs (selection) and do a thing many times (repetition), so a short program can specify a potentially unlimited number of executed statements. |
| lvl:control_flow@L4#0 \| level L4 way predict_a_change: Reordering two statements generally changes the result whenever one reads a variable that the other writes (e.g. swapping x = x + 1 and print(x)); reordering is safe only for statements that touch disjoint variables. Moving a statement into or out of a branch or loop body changes how many times it runs (0, 1 or n). |
| lvl:control_flow@L4#1 \| level L4 way transfer_to_a_new_program: Reason in terms of control flow whenever you must predict output or find why a statement did or did not run: trace which statement is reached next. Straight-line sequence suffices only when every input needs the same steps exactly once; otherwise add a conditional or a loop. |
| lvl:control_flow@L4#2 \| level L4 way diagnose_an_error: rule broken: A local variable must be definitely assigned on every control-flow path before it is read. Kind: compile-time. Trigger class: A variable assigned only inside an if branch (or only in a loop body) and read after it: the compiler reports 'variable might not have been initialized'. |
| lvl:control_flow@L4#3 \| level L4 way diagnose_an_error: rule broken: Statements run in the order written, so an update must come before the statement that needs the new value. Kind: silent logic error. Trigger class: Swapping two dependent statements (e.g. printing before updating, or incrementing the index before using it): output is shifted or off by one pass, with no error. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:control_flow@L0#3 (Score: 0.5)}
{lvl:control_flow@L0#0 (Score: 0.3)}
{lvl:control_flow@L2#1 (Score: 0.2)}
```
Kept by pruning: lvl:control_flow@L0#3 (0.50), lvl:control_flow@L0#0 (0.30), lvl:control_flow@L2#1 (0.20)

Label prompt for level way lvl:control_flow@L0#3 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Applies the sequencing rule (LT) unevenly: e.g. runs a later line before an earlier one, lets an if that was already passed fire again later, or ends the program when a loop or a false if ends instead of continuing with the next statement. (here mixing in: "Statements in a program can run in any order")

Model's reply:
```
Label: shown
Clauses: 2, 3
Confidence: 0.7
```

Label prompt for level way lvl:control_flow@L0#0 (the model saw only the concept, the way's name 'everyday reading' and its text; no level):
> Folk meaning: in everyday instructions a listener reads the whole list at once, does steps in a sensible order and infers what the writer wants. Java instead executes statements one at a time, top to bottom, each only when control reaches it, and nothing later in the program has any effect earlier.

Model's reply:
```
Label: shown
Clauses: 2, 3
Confidence: 0.7
```

Label prompt for level way lvl:control_flow@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> The output is the sequence of print statements that control actually reaches, in the order reached: e.g. TenHellos prints 1st, 2nd, 3rd Hello once each, then 4th to 10th Hello from the loop.

Model's reply:
```
Label: shown
Clauses: 1, 3
Confidence: 0.6
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:control_flow@L0#3 | shown | 0.5 | 0.7 | 0.35 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| lvl:control_flow@L0#0 | shown | 0.3 | 0.7 | 0.21 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| lvl:control_flow@L2#1 | shown | 0.2 | 0.6 | 0.12 | c1, c3 | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |

#### println, misconceptions: 6 edges offered
Edges offered:
| edge |
|---|
| err:println->PrintWithoutFinalNewline \| wrong rule: Ends the output with print and expects it to appear right away. \| right rule: Finish a line of print calls with println; some systems hold print output until a newline is printed. |
| mis:println->IdentifierCaseInsensitive \| wrong rule: System.out.Println and System.out.println name the same method. \| right rule: Java is case-sensitive: names of variables, methods and classes that differ only in capitalization are different names. |
| mis:println->ParenthesesOnlyIfArgument \| wrong rule: println with nothing to print can be written System.out.println; without the parentheses. \| right rule: A method call always needs parentheses, so an empty line is printed with System.out.println();. |
| mis:println->PrintEqualsPrintln \| wrong rule: print and println do the same thing, so print(s1); print(s2); puts s1 and s2 on separate lines. \| right rule: println ends the line after printing and print does not, so print(s1); print(s2); puts s1 and s2 on the same line. |
| mis:println->PrintNewLineFirst \| wrong rule: println(s) prints a newline and then s. \| right rule: println(s) prints s and then the newline. |
| mis:println->PrintedTextIsExecuted \| wrong rule: System.out.println("x is 5"); sets x to 5 (text inside the quotes is run as code). \| right rule: Characters inside double quotes are printed literally; println only displays them and changes no variable. Only an assignment such as x = 5; sets x. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:println->PrintEqualsPrintln (0.60), mis:println->PrintNewLineFirst (0.30), mis:println->PrintedTextIsExecuted (0.10)
```
Kept by pruning: mis:println->PrintEqualsPrintln (0.60), mis:println->PrintNewLineFirst (0.30), mis:println->PrintedTextIsExecuted (0.10)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:println->PrintEqualsPrintln \| wrong rule: print and println do the same thing, so print(s1); print(s2); puts s1 and s2 on separate lines. \| right rule: println ends the line after printing and print does not, so print(s1); print(s2); puts s1 and s2 on the same line. |
| mis:println->PrintNewLineFirst \| wrong rule: println(s) prints a newline and then s. \| right rule: println(s) prints s and then the newline. |
| mis:println->PrintedTextIsExecuted \| wrong rule: System.out.println("x is 5"); sets x to 5 (text inside the quotes is run as code). \| right rule: Characters inside double quotes are printed literally; println only displays them and changes no variable. Only an assignment such as x = 5; sets x. |
Model's reply:
```
mis:println->PrintEqualsPrintln | not applied | clauses: none | confidence: 0.85
mis:println->PrintNewLineFirst | not applied | clauses: none | confidence: 0.85
mis:println->PrintedTextIsExecuted | not applied | clauses: none | confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:println->PrintEqualsPrintln | not applied |  |  | 0.0 |  |  |
| mis:println->PrintNewLineFirst | not applied |  |  | 0.0 |  |  |
| mis:println->PrintedTextIsExecuted | not applied |  |  | 0.0 |  |  |

#### println, level ways: 17 edges offered
Edges offered:
| edge |
|---|
| lvl:println@L0#0 \| level L0 way everyday_reading: Everyday reading: 'println' is just 'print', and 'new line' is something you do before writing. Java rule: println(x) prints x and then ends the line; println() alone just ends the line. |
| lvl:println@L0#3 \| level L0 way inconsistent_trace: Applies the println rule (print, then end line) unevenly: puts the newline before the text on one call and after it on another, or forgets that the next output starts on a new line. (here mixing in: "print and println behave the same (no newline difference)") |
| lvl:println@L1#0 \| level L1 way names_the_element: System.out.println is a library method that prints its argument followed by a line terminator ('print line'). |
| lvl:println@L1#1 \| level L1 way states_the_rule: println(String x) behaves as though it calls print(x) and then println(); println() terminates the current line by writing the line separator string. |
| lvl:println@L2#0 \| level L2 way line_by_line_trace: Evaluate the argument, print its characters at the current position, then write the line separator so the cursor moves to the beginning of the next line. |
| lvl:println@L2#1 \| level L2 way output_prediction: Output is the value's characters followed by a line break: println("Hello, "); println("World!"); prints two lines. |
| lvl:println@L2#2 \| level L2 way one_step_execution: One call appends the text and a line separator to the output; the cursor ends at the start of the next line; no variable changes and nothing is returned. |
| lvl:println@L3#0 \| level L3 way relation_to_neighbour: System.out.println(s); is a statement, so its line of output appears at its own place in the sequence. |
| lvl:println@L3#1 \| level L3 way relation_to_neighbour: println("s") outputs exactly the characters between the quotes and then ends the line. |
| lvl:println@L3#2 \| level L3 way relation_to_neighbour: print leaves the cursor on the line; println ends it; several prints followed by one println make one line. |
| lvl:println@L3#3 \| level L3 way relation_to_neighbour: println prints numbers with all their digits and always ends the line; printf formats each value by its conversion code and ends the line only where the format has %n or \n. |
| lvl:println@L3#4 \| level L3 way purpose_of_the_whole: Purpose: produce one complete line of output, e.g. each result on its own line, or end a line built with print. |
| lvl:println@L4#0 \| level L4 way predict_a_change: Replacing println with print joins the next output onto the same line (a typical logic error); the characters printed stay the same. |
| lvl:println@L4#1 \| level L4 way transfer_to_a_new_program: Use println when each output item belongs on its own line or to end a line; use print to continue a line and printf to control number formatting. |
| lvl:println@L4#2 \| level L4 way diagnose_an_error: rule broken: Finish a line of print calls with println. Kind: silent logic error. Trigger class: output ends with print, so the last line may not be shown or runs into the shell prompt |
| lvl:println@L4#3 \| level L4 way diagnose_an_error: rule broken: Method names are case-sensitive. Kind: compile-time. Trigger class: System.out.Println or system.out.println: cannot find symbol |
| lvl:println@L4#4 \| level L4 way diagnose_an_error: rule broken: Calls need parentheses. Kind: compile-time. Trigger class: System.out.println; without () |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:println@L2#1 (Score: 0.5)}
{lvl:println@L3#0 (Score: 0.3)}
{lvl:println@L3#4 (Score: 0.2)}
```
Kept by pruning: lvl:println@L2#1 (0.50), lvl:println@L3#0 (0.30), lvl:println@L3#4 (0.20)

Label prompt for level way lvl:println@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> Output is the value's characters followed by a line break: println("Hello, "); println("World!"); prints two lines.

Model's reply:
```
Label: shown
Clauses: 1, 3
Confidence: 0.6
```

Label prompt for level way lvl:println@L3#0 (the model saw only the concept, the way's name 'relation to neighbour' and its text; no level):
> System.out.println(s); is a statement, so its line of output appears at its own place in the sequence.

Model's reply:
```
Label: shown
Clauses: 3
Confidence: 0.6
```

Label prompt for level way lvl:println@L3#4 (the model saw only the concept, the way's name 'purpose of the whole' and its text; no level):
> Purpose: produce one complete line of output, e.g. each result on its own line, or end a line built with print.

Model's reply:
```
Label: shown
Clauses: 1, 3
Confidence: 0.6
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:println@L2#1 | shown | 0.5 | 0.6 | 0.3 | c1, c3 | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| lvl:println@L3#0 | shown | 0.3 | 0.6 | 0.18 | c3 | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| lvl:println@L3#4 | shown | 0.2 | 0.6 | 0.12 | c1, c3 | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |

**Kept on a path at this depth** (best N in tiers: held misconception / broken edge, then held error, then correct / confirmed / shown):

pre:for_loop->nested_loops [broken, tier 0, 0.64], pre:loop_control->for_loop [broken, tier 0, 0.51], pre:for_loop->loop_body [broken, tier 0, 0.17], pre:for_loop->loop_scope [broken, tier 0, 0.06]

reached across pre:for_loop->nested_loops, entered next: nested_loops

reached across pre:for_loop->loop_body, entered next: loop_body

reached across pre:for_loop->loop_scope, entered next: loop_scope

**Clause coverage after this depth:**
| clause | text | covered by |
|---|---|---|
| c1 | Three lines | not covered yet |
| c2 | The outer loop finishes first | mis:for_loop->ForLoopPartSkipped |
| c3 | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | mis:for_loop->ForLoopPartSkipped |

**Graph check, not yet read (the walk may not stop while these remain):** own edges of loop_body, own edges of loop_scope, own edges of nested_loops, pre:statement->control_flow, pre:statement->println, pre:string_literal->println

**Figure 5 (the graph after depth 2)**

![Figure 5](figure05.png)
### Depth 3: current concepts 'nested_loops', 'for_loop', 'loop_control', 'loop_body', 'loop_scope', 'control_flow', 'println'

#### nested_loops, misconceptions: 8 edges offered
Edges offered:
| edge |
|---|
| err:nested_loops->BracelessMultiStatementBody \| wrong rule: Indenting several statements under an if or for makes them all part of its body, even without braces. \| right rule: Without braces only the first statement belongs to the if/for; the rest run once, after it. |
| mis:nested_loops->EmptyInnerLoopSkipsRest \| wrong rule: If the inner loop runs zero times, the rest of that outer pass is skipped too. \| right rule: An inner loop that runs zero times just ends; the statements after it in the outer body still run. |
| mis:nested_loops->InconsistentIndentation \| wrong rule: Indents nested blocks inconsistently, so the visual nesting differs from the real nesting. \| right rule: Indent consistently, one level deeper for each nested block, so the layout shows the real nesting; the compiler ignores indentation. |
| mis:nested_loops->InnerCounterOverwritesOuter \| wrong rule: The inner loop count replaces the outer loop count, so the nested loops run as one loop. \| right rule: The two counts multiply: the inner body runs (outer passes) x (inner passes) times. |
| mis:nested_loops->InnerOuterSwapped \| wrong rule: The inner loop's variable changes more slowly than the outer loop's. \| right rule: The inner loop runs all its passes for each single pass of the outer loop. |
| mis:nested_loops->NestedLoopsLockstep \| wrong rule: Nested loops run in step as one loop: both variables change together and both conditions are tested together. \| right rule: For each pass of the outer loop the inner loop runs all its passes; the inner variable changes fastest. |
| mis:nested_loops->NestedLoopsSequential \| wrong rule: The outer loop runs completely, then the inner loop runs completely. \| right rule: The inner loop runs all its passes during each single pass of the outer loop. |
| mis:nested_loops->OuterLoopIgnored \| wrong rule: In nested loops only the inner loop repeats; the outer loop runs once. \| right rule: The outer loop repeats as its own condition says, and the inner loop runs fully on each outer pass. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:nested_loops->NestedLoopsSequential (0.60), mis:nested_loops->InnerCounterOverwritesOuter (0.25), mis:nested_loops->OuterLoopIgnored (0.15)
```
Kept by pruning: mis:nested_loops->NestedLoopsSequential (0.60), mis:nested_loops->InnerCounterOverwritesOuter (0.25), mis:nested_loops->OuterLoopIgnored (0.15)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:nested_loops->NestedLoopsSequential \| wrong rule: The outer loop runs completely, then the inner loop runs completely. \| right rule: The inner loop runs all its passes during each single pass of the outer loop. |
| mis:nested_loops->InnerCounterOverwritesOuter \| wrong rule: The inner loop count replaces the outer loop count, so the nested loops run as one loop. \| right rule: The two counts multiply: the inner body runs (outer passes) x (inner passes) times. |
| mis:nested_loops->OuterLoopIgnored \| wrong rule: In nested loops only the inner loop repeats; the outer loop runs once. \| right rule: The outer loop repeats as its own condition says, and the inner loop runs fully on each outer pass. |
Model's reply:
```
mis:nested_loops->NestedLoopsSequential | held | clauses: 2, 3 | confidence: 0.95
mis:nested_loops->InnerCounterOverwritesOuter | not applied | clauses: none | confidence: 0.75
mis:nested_loops->OuterLoopIgnored | not applied | clauses: none | confidence: 0.75
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:nested_loops->NestedLoopsSequential | held | 0.6 | 0.95 | 0.57 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| mis:nested_loops->InnerCounterOverwritesOuter | not applied |  |  | 0.0 |  |  |
| mis:nested_loops->OuterLoopIgnored | not applied |  |  | 0.0 |  |  |

#### nested_loops, level ways: 17 edges offered
Edges offered:
| edge |
|---|
| lvl:nested_loops@L0#0 \| level L0 way everyday_reading: Folk meaning: 'for each row and each column' sounds like doing two things side by side or one after the other. In Java the inner loop is a statement in the outer body, so it runs all its passes during each single outer pass (like the hands of a clock). |
| lvl:nested_loops@L0#7 \| level L0 way inconsistent_trace: Applies the nested-loop rule (LT) unevenly: resets the inner variable on some outer passes but not others, or advances the outer variable in the middle of the inner loop. (here mixing in: "The outer loop runs completely, then the inner loop runs completely") |
| lvl:nested_loops@L1#0 \| level L1 way names_the_element: Nested loops are a loop inside the body of another loop: the outer loop and the inner loop. |
| lvl:nested_loops@L1#1 \| level L1 way states_the_rule: Because loops are statements, one can be the body of another; the inner loop iterates all the way through for each iteration of the outer loop. |
| lvl:nested_loops@L2#0 \| level L2 way line_by_line_trace: Outer init; outer test; [inner init; inner test; inner body; inner update; ... until the inner test fails]; rest of the outer body; outer update; outer test; ... The inner init runs again on every outer pass. |
| lvl:nested_loops@L2#1 \| level L2 way output_prediction: The inner body runs (outer passes) x (inner passes) times: two loops of 1..10 call printf 100 times; DivisorPattern 3 prints 3 rows of 3 entries, each row ending with its row number. |
| lvl:nested_loops@L2#2 \| level L2 way one_step_execution: One outer pass resets the inner variable, runs the complete inner loop, executes the rest of the outer body once, and then updates the outer variable once. |
| lvl:nested_loops@L3#0 \| level L3 way relation_to_neighbour: Usually both loops are for loops: the inner for runs its complete set of passes for every single pass of the outer for. |
| lvl:nested_loops@L3#1 \| level L3 way relation_to_neighbour: Matrix addition uses two nested loops over i and j; matrix multiplication adds a third, innermost loop over k for each dot product. |
| lvl:nested_loops@L3#2 \| level L3 way relation_to_neighbour: A plain break ends only the innermost loop; a labeled break is needed when the inner loop must end the outer one too. |
| lvl:nested_loops@L3#3 \| level L3 way relation_to_neighbour: Which loop is outside decides the visiting order: row index outside gives row-major order, column index outside gives column-major order. |
| lvl:nested_loops@L3#4 \| level L3 way purpose_of_the_whole: Plan: process every pair (i, j) - two-dimensional tables, every row x column of a matrix, every pair of elements; the outer variable is a slow stepper, the inner a fast stepper. |
| lvl:nested_loops@L4#0 \| level L4 way predict_a_change: Swapping the outer and inner loops visits the same pairs in a different order (rows vs columns), which preserves a sum but changes printed layout; moving a statement between the inner body and the outer body changes its count from (outer x inner) to outer. |
| lvl:nested_loops@L4#1 \| level L4 way transfer_to_a_new_program: Use nested loops when the work ranges over all combinations of two indices (tables, matrices, pairs); expect running time proportional to the product of the pass counts, and prefer a single loop when the second index is a function of the first. |
| lvl:nested_loops@L4#2 \| level L4 way diagnose_an_error: rule broken: Without braces, the body of an if, while or for is exactly one statement; indentation does not add statements to it. Kind: silent logic error. Trigger class: A braceless header followed by two or more indented statements: only the first is controlled, the others run once, after the if/loop, on every input. |
| lvl:nested_loops@L4#3 \| level L4 way diagnose_an_error: rule broken: The inner loop needs its own loop variable. Kind: compile-time. Trigger class: Redeclaring the outer variable in the inner header (for (int i ...) { for (int i ...) }): 'variable i is already defined'. |
| lvl:nested_loops@L4#4 \| level L4 way diagnose_an_error: rule broken: The inner loop must update its own variable. Kind: silent logic error. Trigger class: Writing for (int j = 0; j < N; i++) in the inner loop: j never changes, so the inner loop is infinite (or i runs out of range). |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:nested_loops@L0#7 (Score: 0.5)}
{lvl:nested_loops@L0#0 (Score: 0.3)}
{lvl:nested_loops@L2#1 (Score: 0.2)}
```
Kept by pruning: lvl:nested_loops@L0#7 (0.50), lvl:nested_loops@L0#0 (0.30), lvl:nested_loops@L2#1 (0.20)

Label prompt for level way lvl:nested_loops@L0#7 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Applies the nested-loop rule (LT) unevenly: resets the inner variable on some outer passes but not others, or advances the outer variable in the middle of the inner loop. (here mixing in: "The outer loop runs completely, then the inner loop runs completely")

Model's reply:
```
Label: shown
Clauses: 2, 3
Confidence: 0.7
```

Label prompt for level way lvl:nested_loops@L0#0 (the model saw only the concept, the way's name 'everyday reading' and its text; no level):
> Folk meaning: 'for each row and each column' sounds like doing two things side by side or one after the other. In Java the inner loop is a statement in the outer body, so it runs all its passes during each single outer pass (like the hands of a clock).

Model's reply:
```
Label: shown
Clauses: 2, 3
Confidence: 0.9
```

Label prompt for level way lvl:nested_loops@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> The inner body runs (outer passes) x (inner passes) times: two loops of 1..10 call printf 100 times; DivisorPattern 3 prints 3 rows of 3 entries, each row ending with its row number.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.8
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:nested_loops@L0#7 | shown | 0.5 | 0.7 | 0.35 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| lvl:nested_loops@L0#0 | shown | 0.3 | 0.9 | 0.27 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| lvl:nested_loops@L2#1 | not applied |  |  | 0.0 |  |  |

#### loop_control, concept edges: 2 edges offered
Edges offered:
| edge |
|---|
| pre:loop_control->for_loop \| rule: The for header is exactly the three control slots, and the body runs after the test and before the update on each pass. |
| pre:loop_control->increment_operators \| rule: The loop update i = i + 1 comes first; ++ (and +=) are then introduced as shorthand for that update. |
| pre:loop_variable->loop_control \| rule: The init, condition and update all act on the loop variable: the init sets it, the condition tests it, the update changes it. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
pre:loop_variable->loop_control (0.80), pre:loop_control->increment_operators (0.20)
```
Kept by pruning: pre:loop_variable->loop_control (0.80), pre:loop_control->increment_operators (0.20)

Label prompt for concept edge pre:loop_variable->loop_control (loop_variable -> loop_control, prerequisite):
> The init, condition and update all act on the loop variable: the init sets it, the condition tests it, the update changes it.

Model's reply:
```
Label: confirmed
Clauses: 3
Confidence: 0.6
```

Label prompt for concept edge pre:loop_control->increment_operators (loop_control -> increment_operators, prerequisite):
> The loop update i = i + 1 comes first; ++ (and +=) are then introduced as shorthand for that update.

Model's reply:
```
Label: not applied
Clauses: none
Confidence: 0.7
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:loop_variable->loop_control | confirmed | 0.8 | 0.6 | 0.48 | c3 | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| pre:loop_control->increment_operators | not applied |  |  | 0.0 |  |  |

#### loop_body, misconceptions: 12 edges offered
Edges offered:
| edge |
|---|
| err:loop_body->BracelessMultiStatementBody \| wrong rule: Indenting several statements under an if or for makes them all part of its body, even without braces. \| right rule: Without braces only the first statement belongs to the if/for; the rest run once, after it. |
| mis:loop_body->ComputerFillsInIntent \| wrong rule: The computer fills in omitted steps or values that the programmer meant, e.g. REPEAT 4 [FORWARD 30] draws a square without a turn command. \| right rule: The computer executes only what the code states; every needed step and value must be written. |
| mis:loop_body->ForLoopPartSkipped \| wrong rule: A pass of a loop may leave out one of its parts (the condition test, the body, the update, or the initialization). \| right rule: Each pass of a for loop runs test, then body, then update, every time; the initialization runs once, at the start. |
| mis:loop_body->IfInLoopUnchecked \| wrong rule: An if inside a loop body runs its branch on every pass without testing its condition. \| right rule: The if condition is tested again on every pass, with the current values, so the branch runs only on passes where it is true. |
| mis:loop_body->LoopBodyAlwaysOnce \| wrong rule: The loop body always runs at least once, even if the condition is false at the start. \| right rule: A while or for body runs zero times when the condition is false at the start. |
| mis:loop_body->LoopBodyStatementOutsideLoop \| wrong rule: Some statements inside the braces of a loop body are not repeated; they run once, before or after the loop. \| right rule: Every statement inside the loop braces is part of the body and runs on every pass. |
| mis:loop_body->LoopCountIsStatementCount \| wrong rule: The repeat count of a counted loop says how many of the body statements are executed. \| right rule: The repeat count is how many times the whole body runs; every pass runs all body statements. |
| mis:loop_body->LoopOutputSameEachPass \| wrong rule: A loop prints the same thing on every pass; the changing loop variable does not affect what the body does. \| right rule: The body uses the current value of the loop variable, so its output can differ on each pass. |
| mis:loop_body->LoopRunsStatementsSeparately \| wrong rule: A loop runs its first body statement all N times, then its second body statement all N times. \| right rule: Each pass runs the whole body in order, so the body statements interleave: S1, S2, S1, S2, ... |
| mis:loop_body->StateChangeIgnoredAcrossPasses \| wrong rule: Values changed by one pass of a loop (other than the loop variable) do not affect later passes. \| right rule: Each pass starts from the state left by the previous pass, so a change in one pass carries into the next ones. |
| mis:loop_body->StatementAfterLoopInBody \| wrong rule: A statement written after the loop is repeated with the loop body. \| right rule: The body ends at the closing brace (or after its single statement); code after it runs once, after the loop ends. |
| mis:loop_body->WhileStopsMidBody \| wrong rule: The loop stops the moment the condition becomes false, even in the middle of the body. \| right rule: The condition is tested only before each pass, so the whole body finishes before the loop can stop. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:loop_body->LoopBodyStatementOutsideLoop (0.50), mis:loop_body->ForLoopPartSkipped (0.30), mis:loop_body->LoopRunsStatementsSeparately (0.20)
```
Kept by pruning: mis:loop_body->LoopBodyStatementOutsideLoop (0.50), mis:loop_body->ForLoopPartSkipped (0.30), mis:loop_body->LoopRunsStatementsSeparately (0.20)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:loop_body->LoopBodyStatementOutsideLoop \| wrong rule: Some statements inside the braces of a loop body are not repeated; they run once, before or after the loop. \| right rule: Every statement inside the loop braces is part of the body and runs on every pass. |
| mis:loop_body->ForLoopPartSkipped \| wrong rule: A pass of a loop may leave out one of its parts (the condition test, the body, the update, or the initialization). \| right rule: Each pass of a for loop runs test, then body, then update, every time; the initialization runs once, at the start. |
| mis:loop_body->LoopRunsStatementsSeparately \| wrong rule: A loop runs its first body statement all N times, then its second body statement all N times. \| right rule: Each pass runs the whole body in order, so the body statements interleave: S1, S2, S1, S2, ... |
Model's reply:
```
mis:loop_body->LoopBodyStatementOutsideLoop | held | clauses: 1, 2, 3 | confidence: 0.85
mis:loop_body->ForLoopPartSkipped | not applied | clauses: none | confidence: 0.7
mis:loop_body->LoopRunsStatementsSeparately | not applied | clauses: none | confidence: 0.75
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:loop_body->LoopBodyStatementOutsideLoop | held | 0.5 | 0.85 | 0.42 | c1, c2, c3 | Three lines; The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| mis:loop_body->ForLoopPartSkipped | not applied |  |  | 0.0 |  |  |
| mis:loop_body->LoopRunsStatementsSeparately | not applied |  |  | 0.0 |  |  |

#### loop_body, level ways: 14 edges offered
Edges offered:
| edge |
|---|
| lvl:loop_body@L0#0 \| level L0 way everyday_reading: Folk meaning: an everyday 'repeat this' instruction can be read loosely - a helper fills in missing steps, does each step N times in turn, or stops partway. A Java loop body is exactly the statements inside its braces (or one statement), all executed in order on every pass. |
| lvl:loop_body@L0#6 \| level L0 way inconsistent_trace: Applies the body rule (LT) unevenly: runs all body statements on some passes but only some on others, includes the statement after the loop on some passes, or uses stale values from the first pass later. (here mixing in: "The loop body always runs at least once, even if the condition is false at the start") |
| lvl:loop_body@L1#0 \| level L1 way names_the_element: The loop body is the statement block that a loop repeats on each pass. |
| lvl:loop_body@L1#1 \| level L1 way states_the_rule: The body is the block after the loop header; braces are optional only if it is a single statement, in which case only that one statement is repeated. |
| lvl:loop_body@L2#0 \| level L2 way line_by_line_trace: On each pass the body's statements run in order, first to last, after a true test (and, in a for loop, before the update); the next pass starts from the state left by this one. |
| lvl:loop_body@L2#1 \| level L2 way output_prediction: The body's output on each pass depends on the current values, so successive passes can print different lines: TenHellos prints '4th Hello', '5th Hello', ...; the number of executions of each body statement equals the number of passes. |
| lvl:loop_body@L2#2 \| level L2 way one_step_execution: One pass of the body changes exactly the variables it assigns (e.g. print i and v, double v, increment i) and these changes carry into the next test and pass. |
| lvl:loop_body@L3#0 \| level L3 way relation_to_neighbour: The accumulator update (sum += x) is placed inside the body so it runs once per pass, while the accumulator is declared and initialized before the loop. |
| lvl:loop_body@L3#1 \| level L3 way relation_to_neighbour: count++ sits inside the body, often under an if, so it runs once on each pass that matches; the counter is declared before the loop. |
| lvl:loop_body@L3#2 \| level L3 way relation_to_neighbour: The body is what the while repeats; the condition is tested only between complete executions of it. |
| lvl:loop_body@L3#3 \| level L3 way purpose_of_the_whole: Plan: the body is the 'process one item / one step' part of a loop plan: each pass performs the same process on a new value (next element, next bet, next term). |
| lvl:loop_body@L4#0 \| level L4 way predict_a_change: Moving a statement from inside the body to after the loop changes it from running once per pass to once in total (and vice versa); reordering two body statements that share a variable changes the results (e.g. increment before vs after use). |
| lvl:loop_body@L4#1 \| level L4 way diagnose_an_error: rule broken: Without braces, the body of an if, while or for is exactly one statement; indentation does not add statements to it. Kind: silent logic error. Trigger class: A braceless header followed by two or more indented statements: only the first is controlled, the others run once, after the if/loop, on every input. |
| lvl:loop_body@L4#2 \| level L4 way transfer_to_a_new_program: Put inside the body exactly what must happen once per pass; put set-up before the loop and final reporting after it. Always brace a multi-statement body. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:loop_body@L0#6 (Score: 0.45)}
{lvl:loop_body@L0#0 (Score: 0.3)}
{lvl:loop_body@L2#1 (Score: 0.25)}
```
Kept by pruning: lvl:loop_body@L0#6 (0.45), lvl:loop_body@L0#0 (0.30), lvl:loop_body@L2#1 (0.25)

Label prompt for level way lvl:loop_body@L0#6 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Applies the body rule (LT) unevenly: runs all body statements on some passes but only some on others, includes the statement after the loop on some passes, or uses stale values from the first pass later. (here mixing in: "The loop body always runs at least once, even if the condition is false at the start")

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.6
```

Label prompt for level way lvl:loop_body@L0#0 (the model saw only the concept, the way's name 'everyday reading' and its text; no level):
> Folk meaning: an everyday 'repeat this' instruction can be read loosely - a helper fills in missing steps, does each step N times in turn, or stops partway. A Java loop body is exactly the statements inside its braces (or one statement), all executed in order on every pass.

Model's reply:
```
Label: shown
Clauses: 2, 3
Confidence: 0.8
```

Label prompt for level way lvl:loop_body@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> The body's output on each pass depends on the current values, so successive passes can print different lines: TenHellos prints '4th Hello', '5th Hello', ...; the number of executions of each body statement equals the number of passes.

Model's reply:
```
Label: shown
Clauses: 1, 3
Confidence: 0.7
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:loop_body@L0#6 | not applied |  |  | 0.0 |  |  |
| lvl:loop_body@L0#0 | shown | 0.3 | 0.8 | 0.24 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| lvl:loop_body@L2#1 | shown | 0.25 | 0.7 | 0.17 | c1, c3 | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |

#### loop_scope, misconceptions: 5 edges offered
Edges offered:
| edge |
|---|
| err:loop_scope->AccumulatorDeclaredInLoop \| wrong rule: Declaring sum inside the loop body keeps a running total across passes. \| right rule: The accumulator must be declared and initialised before the loop. |
| mis:loop_scope->ForVariableScopeBeyondLoop \| wrong rule: i declared in the for header can still be used after the loop has ended. \| right rule: i declared in the for header exists only inside the loop, so using it after the loop does not compile. |
| mis:loop_scope->LocalVarOutsideScope \| wrong rule: A variable declared inside a block can be used anywhere in the program, like a global. \| right rule: A local variable exists only from its declaration to the end of its block. |
| mis:loop_scope->LoopBodyScopeImpliesLoopLifetime \| wrong rule: A variable declared inside the loop body keeps its value from one pass to the next. \| right rule: A variable declared inside the body is created anew on every pass and does not keep the previous pass's value. |
| mis:loop_scope->LoopVariableLifetimeScope \| wrong rule: The for variable i is a new variable on every pass rather than one variable changed by the update. \| right rule: The for variable i is created once by the init, and the update changes that same variable on every pass. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{mis:loop_scope->ForVariableScopeBeyondLoop (Score: 0.6)}
{mis:loop_scope->LocalVarOutsideScope (Score: 0.25)}
{mis:loop_scope->LoopVariableLifetimeScope (Score: 0.15)}
```
Kept by pruning: mis:loop_scope->ForVariableScopeBeyondLoop (0.60), mis:loop_scope->LocalVarOutsideScope (0.25), mis:loop_scope->LoopVariableLifetimeScope (0.15)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:loop_scope->ForVariableScopeBeyondLoop \| wrong rule: i declared in the for header can still be used after the loop has ended. \| right rule: i declared in the for header exists only inside the loop, so using it after the loop does not compile. |
| mis:loop_scope->LocalVarOutsideScope \| wrong rule: A variable declared inside a block can be used anywhere in the program, like a global. \| right rule: A local variable exists only from its declaration to the end of its block. |
| mis:loop_scope->LoopVariableLifetimeScope \| wrong rule: The for variable i is a new variable on every pass rather than one variable changed by the update. \| right rule: The for variable i is created once by the init, and the update changes that same variable on every pass. |
Model's reply:
```
mis:loop_scope->ForVariableScopeBeyondLoop | held | clauses: 2, 3 | confidence: 0.6
mis:loop_scope->LocalVarOutsideScope | not applied | clauses: none | confidence: 0.75
mis:loop_scope->LoopVariableLifetimeScope | not applied | clauses: none | confidence: 0.8
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:loop_scope->ForVariableScopeBeyondLoop | held | 0.6 | 0.6 | 0.36 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| mis:loop_scope->LocalVarOutsideScope | not applied |  |  | 0.0 |  |  |
| mis:loop_scope->LoopVariableLifetimeScope | not applied |  |  | 0.0 |  |  |

#### loop_scope, level ways: 14 edges offered
Edges offered:
| edge |
|---|
| lvl:loop_scope@L0#0 \| level L0 way everyday_reading: Folk meaning: once a name is introduced ('let i be ...') it can be referred to for the rest of the text, and something introduced inside a repeated step keeps existing across repetitions. In Java a local variable exists only from its declaration to the end of its block, and a for header counts as part of the loop's block. |
| lvl:loop_scope@L0#5 \| level L0 way inconsistent_trace: Applies the scope rule (LT) unevenly: lets a body variable keep its value from the last pass on some passes but not others, or uses the for variable after the loop in one place but not another. (here mixing in: "a variable declared in the loop body lives across all iterations") |
| lvl:loop_scope@L1#0 \| level L1 way names_the_element: Scope is the part of the program where a variable is defined (can be used); for loops, the header is in the same block as the body. |
| lvl:loop_scope@L1#1 \| level L1 way states_the_rule: A local variable's scope is the rest of the block after its declaration; a variable declared in a for initializer is in scope in the header and the body only; redeclaring a local variable inside its scope is a compile-time error. |
| lvl:loop_scope@L2#0 \| level L2 way line_by_line_trace: Each time control enters a block, its declarations run in order, creating and (if initialized) setting those variables; when the block ends they cease to exist. For a loop body this happens on every pass; for the for-header variable once, at the init. |
| lvl:loop_scope@L2#1 \| level L2 way output_prediction: A variable declared inside the body starts from its initializer on every pass (int s = 0; inside the body gives s = 0 at the start of each pass); a for-header variable cannot be printed after the loop (compile error). |
| lvl:loop_scope@L2#2 \| level L2 way one_step_execution: Entering the body creates its local variables afresh; leaving it removes them; values left in variables declared before the loop persist into the next pass. |
| lvl:loop_scope@L3#0 \| level L3 way relation_to_neighbour: A variable declared in the for header exists only from its declaration to the end of the for statement, so for and while versions differ: in the while version the counter is still available after the loop. |
| lvl:loop_scope@L3#1 \| level L3 way purpose_of_the_whole: Plan: keep each variable as local as possible - declare a loop index in the for header when it is not needed afterwards, and declare variables that must survive the loop (accumulators, counters, loop variables needed later) before it. |
| lvl:loop_scope@L4#0 \| level L4 way predict_a_change: Moving a declaration from before the loop into the body re-initializes the variable every pass and makes it unusable after the loop; moving a for-header declaration out (int i; for (i = 0; ...)) keeps the loop's behaviour and makes i usable after it. |
| lvl:loop_scope@L4#1 \| level L4 way diagnose_an_error: rule broken: A variable declared in the body is in scope only to the end of the body and is re-declared and re-initialized on every pass. Kind: silent logic error. Trigger class: for (...) { int sum = 0; sum += a[i]; }: sum restarts at 0 each pass, so only the last term survives; printing sum after the loop is a compile-time error (cannot find symbol) because sum is out of scope. |
| lvl:loop_scope@L4#2 \| level L4 way transfer_to_a_new_program: Declare the index in the for header when it is used only inside the loop; declare it before the loop when its final value is needed (as in the break-based primality test) and declare running results before the loop. |
| lvl:loop_scope@L4#3 \| level L4 way diagnose_an_error: rule broken: A for-header variable is out of scope after the loop. Kind: compile-time. Trigger class: Using i after for (int i = ...) {...}: 'cannot find symbol'. |
| lvl:loop_scope@L4#4 \| level L4 way diagnose_an_error: rule broken: A local variable cannot be redeclared in its own scope. Kind: compile-time. Trigger class: Declaring int i inside a for (int i ...) body, or an inner for (int i ...) inside an outer for (int i ...): 'variable i is already defined'. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:loop_scope@L0#0 (Score: 0.45)}
{lvl:loop_scope@L4#3 (Score: 0.35)}
{lvl:loop_scope@L0#5 (Score: 0.2)}
```
Kept by pruning: lvl:loop_scope@L0#0 (0.45), lvl:loop_scope@L4#3 (0.35), lvl:loop_scope@L0#5 (0.20)

Label prompt for level way lvl:loop_scope@L0#0 (the model saw only the concept, the way's name 'everyday reading' and its text; no level):
> Folk meaning: once a name is introduced ('let i be ...') it can be referred to for the rest of the text, and something introduced inside a repeated step keeps existing across repetitions. In Java a local variable exists only from its declaration to the end of its block, and a for header counts as part of the loop's block.

Model's reply:
```
Label: shown
Clauses: 3
Confidence: 0.6
```

Label prompt for level way lvl:loop_scope@L4#3 (the model saw only the concept, the way's name 'diagnose an error' and its text; no level):
> rule broken: A for-header variable is out of scope after the loop. Kind: compile-time. Trigger class: Using i after for (int i = ...) {...}: 'cannot find symbol'.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:loop_scope@L0#5 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Applies the scope rule (LT) unevenly: lets a body variable keep its value from the last pass on some passes but not others, or uses the for variable after the loop in one place but not another. (here mixing in: "a variable declared in the loop body lives across all iterations")

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.7
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:loop_scope@L0#0 | shown | 0.45 | 0.6 | 0.27 | c3 | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| lvl:loop_scope@L4#3 | not applied |  |  | 0.0 |  |  |
| lvl:loop_scope@L0#5 | not applied |  |  | 0.0 |  |  |

#### control_flow, concept edges: 1 edges offered
Edges offered:
| edge |
|---|
| pre:statement->control_flow \| rule: Control flow is the order in which statements run. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{pre:statement->control_flow (Score: 1.0)}
```
Kept by pruning: pre:statement->control_flow (1.00)

Label prompt for concept edge pre:statement->control_flow (statement -> control_flow, prerequisite):
> Control flow is the order in which statements run.

Model's reply:
```
Label: broken
Clauses: 2, 3
Confidence: 0.8
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:statement->control_flow | broken | 1.0 | 0.8 | 0.8 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |

#### println, concept edges: 2 edges offered
Edges offered:
| edge |
|---|
| pre:statement->println \| rule: System.out.println(s); is itself a statement, so its line of output appears at its own place in the sequence. |
| pre:string_literal->println \| rule: println("s") outputs exactly the characters between the quotes and then ends the line. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{pre:statement->println (Score: 0.8)}
{pre:string_literal->println (Score: 0.2)}
```
Kept by pruning: pre:statement->println (0.80), pre:string_literal->println (0.20)

Label prompt for concept edge pre:statement->println (statement -> println, prerequisite):
> System.out.println(s); is itself a statement, so its line of output appears at its own place in the sequence.

Model's reply:
```
Label: broken
Clauses: 1, 3
Confidence: 0.6
```

Label prompt for concept edge pre:string_literal->println (string_literal -> println, prerequisite):
> println("s") outputs exactly the characters between the quotes and then ends the line.

Model's reply:
```
Label: confirmed
Clauses: 3
Confidence: 0.6
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:statement->println | broken | 0.8 | 0.6 | 0.48 | c1, c3 | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| pre:string_literal->println | confirmed | 0.2 | 0.6 | 0.12 | c3 | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |

**Kept on a path at this depth** (best N in tiers: held misconception / broken edge, then held error, then correct / confirmed / shown):

pre:statement->control_flow [broken, tier 0, 0.80], mis:nested_loops->NestedLoopsSequential [held, tier 0, 0.57], pre:statement->println [broken, tier 0, 0.48], mis:loop_body->LoopBodyStatementOutsideLoop [held, tier 0, 0.42], mis:loop_scope->ForVariableScopeBeyondLoop [held, tier 0, 0.36]

reached across pre:statement->control_flow, entered next: statement

**Clause coverage after this depth:**
| clause | text | covered by |
|---|---|---|
| c1 | Three lines | mis:loop_body->LoopBodyStatementOutsideLoop |
| c2 | The outer loop finishes first | mis:for_loop->ForLoopPartSkipped |
| c3 | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | mis:for_loop->ForLoopPartSkipped |

**Graph check, not yet read (the walk may not stop while these remain):** own edges of statement

**Figure 6 (the graph after depth 3)**

![Figure 6](figure06.png)
### Depth 4: current concepts 'statement'

#### statement, misconceptions: 5 edges offered
Edges offered:
| edge |
|---|
| err:statement->StatementNeedsNoTerminator \| wrong rule: A statement still compiles and runs if the semicolon at its end is left out, or if a stray ; is put right after if (c) or for (...). \| right rule: Each statement must end with ;, otherwise javac reports ';' expected. |
| mis:statement->ComputerFillsInIntent \| wrong rule: The computer fills in omitted steps or values that the programmer meant, e.g. REPEAT 4 [FORWARD 30] draws a square without a turn command. \| right rule: The computer executes only what the code states; every needed step and value must be written. |
| mis:statement->ProgramLooksAhead \| wrong rule: The program knows or sees what later lines will do and acts on those goals at the current line, e.g. it stops 'at ten' because it wants to draw a box. \| right rule: Each statement executes using only the current state; later lines have no effect until control reaches them. |
| mis:statement->StatementsRunInAnyOrder \| wrong rule: Statements in a program can run in any order. \| right rule: Without control statements, statements run once each, top to bottom. |
| mis:statement->WhitespaceIsSignificant \| wrong rule: Line breaks and indentation change what the program does, so each statement must be on its own line. \| right rule: Java compilers consider all whitespace in program text (outside quotation marks) to be equivalent; layout is for human readers. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:statement->StatementsRunInAnyOrder (0.70), mis:statement->ProgramLooksAhead (0.20), mis:statement->WhitespaceIsSignificant (0.10)
```
Kept by pruning: mis:statement->StatementsRunInAnyOrder (0.70), mis:statement->ProgramLooksAhead (0.20), mis:statement->WhitespaceIsSignificant (0.10)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:statement->StatementsRunInAnyOrder \| wrong rule: Statements in a program can run in any order. \| right rule: Without control statements, statements run once each, top to bottom. |
| mis:statement->ProgramLooksAhead \| wrong rule: The program knows or sees what later lines will do and acts on those goals at the current line, e.g. it stops 'at ten' because it wants to draw a box. \| right rule: Each statement executes using only the current state; later lines have no effect until control reaches them. |
| mis:statement->WhitespaceIsSignificant \| wrong rule: Line breaks and indentation change what the program does, so each statement must be on its own line. \| right rule: Java compilers consider all whitespace in program text (outside quotation marks) to be equivalent; layout is for human readers. |
Model's reply:
```
mis:statement->StatementsRunInAnyOrder | not applied | clauses: none | confidence: 0.7
mis:statement->ProgramLooksAhead | not applied | clauses: none | confidence: 0.85
mis:statement->WhitespaceIsSignificant | not applied | clauses: none | confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:statement->StatementsRunInAnyOrder | not applied |  |  | 0.0 |  |  |
| mis:statement->ProgramLooksAhead | not applied |  |  | 0.0 |  |  |
| mis:statement->WhitespaceIsSignificant | not applied |  |  | 0.0 |  |  |

#### statement, level ways: 19 edges offered
Edges offered:
| edge |
|---|
| lvl:statement@L0#0 \| level L0 way everyday_reading: Everyday reading: a program is read like a text or a list of wishes that the computer takes in all at once, so line order, goals of later lines or what the writer meant can matter. Java rule: each statement is one step, ended by ;, executed once, in written order, using only the current state. |
| lvl:statement@L0#2 \| level L0 way inconsistent_trace: Applies the sequencing rule (LT) unevenly: executes the statements in order on some lines but lets a later statement's effect leak into an earlier one, or skips or repeats a statement on one pass. (here mixing in: "The computer fills in omitted steps or values that the programmer meant, e.g. REPEAT 4 [FORWARD 30] draws a square without a turn command") |
| lvl:statement@L1#0 \| level L1 way names_the_element: A statement is one step of a program (a declaration, assignment, method call such as System.out.println(...), or control statement), ended by a semicolon when it is a simple statement. |
| lvl:statement@L1#1 \| level L1 way states_the_rule: A method body is a sequence of statements enclosed in braces and each followed by a semicolon; statements are executed for their effect and have no value; an expression statement evaluates its expression and discards any value. |
| lvl:statement@L2#0 \| level L2 way line_by_line_trace: Statements run one at a time, from first to last; each completes (normally or abruptly) before the next starts. |
| lvl:statement@L2#1 \| level L2 way output_prediction: A statement has no value; its result is its effect: a changed variable, printed output, or a change of control flow. |
| lvl:statement@L2#2 \| level L2 way one_step_execution: Executing one statement applies its single effect to the current state (e.g. prints one string, or stores one value) and then control moves to the next statement. |
| lvl:statement@L3#0 \| level L3 way relation_to_neighbour: Statements live in main's body; running the program executes main's statements in order. |
| lvl:statement@L3#1 \| level L3 way relation_to_neighbour: A block groups a sequence of statements in braces; the body of main() is such a block. |
| lvl:statement@L3#2 \| level L3 way relation_to_neighbour: System.out.print(s); is a statement, so its output appears exactly at its place in the sequence, with no newline. |
| lvl:statement@L3#3 \| level L3 way relation_to_neighbour: System.out.println(s); is a statement, so its line of output appears at its place in the sequence; swapping two println statements swaps the output lines. |
| lvl:statement@L3#4 \| level L3 way relation_to_neighbour: Control flow is the order in which statements run; without control statements it is straight-line, each once in order. |
| lvl:statement@L3#5 \| level L3 way relation_to_neighbour: A lone ; is the empty statement, which does nothing; it can fill a for-loop part or, by accident, end an if or loop header. |
| lvl:statement@L3#6 \| level L3 way relation_to_neighbour: An algorithm becomes a program as a sequence of statements, executed once each in the order given. |
| lvl:statement@L3#7 \| level L3 way purpose_of_the_whole: Purpose: a statement is the unit of action; a program is characterized by its sequence of statements, each doing one step of the algorithm. |
| lvl:statement@L4#0 \| level L4 way predict_a_change: Swapping two statements that touch the same variable or the output generally changes the result; swapping independent statements, or changing only whitespace between them, keeps it. |
| lvl:statement@L4#1 \| level L4 way diagnose_an_error: rule broken: Each simple statement must end with ;. Kind: compile-time. Trigger class: omitting the ; at the end of a statement: javac reports ';' expected (often at the end of that line or the next) |
| lvl:statement@L4#2 \| level L4 way transfer_to_a_new_program: Write one statement per step and order them as the algorithm requires; straight-line sequence suffices when every step runs once, otherwise use conditionals or loops. |
| lvl:statement@L4#3 \| level L4 way diagnose_an_error: rule broken: A ; directly after an if/while/for header is an empty statement. Kind: silent logic error. Trigger class: if (x > 0); or while (cond); followed by the intended body |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:statement@L0#2 (Score: 0.5)}
{lvl:statement@L0#0 (Score: 0.25)}
{lvl:statement@L3#4 (Score: 0.25)}
```
Kept by pruning: lvl:statement@L0#2 (0.50), lvl:statement@L0#0 (0.25), lvl:statement@L3#4 (0.25)

Label prompt for level way lvl:statement@L0#2 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Applies the sequencing rule (LT) unevenly: executes the statements in order on some lines but lets a later statement's effect leak into an earlier one, or skips or repeats a statement on one pass. (here mixing in: "The computer fills in omitted steps or values that the programmer meant, e.g. REPEAT 4 [FORWARD 30] draws a square without a turn command")

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.6
```

Label prompt for level way lvl:statement@L0#0 (the model saw only the concept, the way's name 'everyday reading' and its text; no level):
> Everyday reading: a program is read like a text or a list of wishes that the computer takes in all at once, so line order, goals of later lines or what the writer meant can matter. Java rule: each statement is one step, ended by ;, executed once, in written order, using only the current state.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.6
```

Label prompt for level way lvl:statement@L3#4 (the model saw only the concept, the way's name 'relation to neighbour' and its text; no level):
> Control flow is the order in which statements run; without control statements it is straight-line, each once in order.

Model's reply:
```
Label: shown
Clauses: 2, 3
Confidence: 0.6
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:statement@L0#2 | not applied |  |  | 0.0 |  |  |
| lvl:statement@L0#0 | not applied |  |  | 0.0 |  |  |
| lvl:statement@L3#4 | shown | 0.25 | 0.6 | 0.15 | c2, c3 | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |

**Kept on a path at this depth** (best N in tiers: held misconception / broken edge, then held error, then correct / confirmed / shown):

lvl:statement@L3#4 [shown, tier 2, 0.15]

**Clause coverage after this depth:**
| clause | text | covered by |
|---|---|---|
| c1 | Three lines | mis:loop_body->LoopBodyStatementOutsideLoop |
| c2 | The outer loop finishes first | mis:for_loop->ForLoopPartSkipped |
| c3 | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | mis:for_loop->ForLoopPartSkipped |

**Figure 7 (the graph after depth 4)**

![Figure 7](figure07.png)
Held set after the one-fault rules: mis:nested_loops->NestedLoopsSequential, mis:loop_body->LoopBodyStatementOutsideLoop, mis:loop_scope->ForVariableScopeBeyondLoop, mis:for_loop->ForLoopPartSkipped; filed at the dependent concept: mis:loop_control->ForLoopPartSkipped

Shadows (a held label on a prerequisite or a generic concept that rests on the words of a stronger label at the specific concept is one wrong sentence read twice; it marks no root cause and sets no level): mis:for_loop->ForLoopPartSkipped (shadow of mis:loop_scope->ForVariableScopeBeyondLoop), mis:loop_control->ForLoopPartSkipped (shadow of mis:loop_body->LoopBodyStatementOutsideLoop)


## 4. Diagnosis (rules over the labelled graph, no model)
Root-cause walk: held on ['loop_body', 'loop_scope', 'nested_loops'], broken at [] (consequences: pre:for_loop->nested_loops, pre:loop_control->for_loop, pre:for_loop->loop_body, pre:for_loop->loop_scope, pre:statement->control_flow, pre:statement->println), prerequisites {'increment_operators': ['assignment', 'loop_control'], 'boolean_expression': ['comparison_operators'], 'comparison_operators': ['expression'], 'loop_body': ['for_loop'], 'loop_scope': ['for_loop'], 'nested_loops': ['for_loop'], 'for_loop': ['loop_control'], 'loop_control': ['loop_variable'], 'control_flow': ['statement'], 'println': ['statement', 'string_literal'], 'string_conversion': ['string_concatenation']}
- control_flow: L2 via output_prediction "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
- loop_body: L0 via misconception "Three lines; The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
- loop_control: L2 via output_prediction "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
- loop_scope: L0 via misconception "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
- nested_loops: L0 via misconception "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
- println: L3 via relation_to_neighbour "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
- statement: L3 via relation_to_neighbour "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
- string_concatenation: L3 via purpose_of_the_whole "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"

**Stop:** sufficient at depth 4; prompts {'prune': 24, 'label': 48}, fresh model calls 0, cache hits 72

### Held misconceptions
| concept | misconception | kind | quote | score | shares_evidence_with |
|---|---|---|---|---|---|
| nested_loops | NestedLoopsSequential | misconception | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | 0.57 |  |
| loop_body | LoopBodyStatementOutsideLoop | misconception | Three lines; The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | 0.425 |  |
| loop_scope | ForVariableScopeBeyondLoop | misconception | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | 0.36 |  |

### Shadowed labels (held in the reading, folded into the label they shadow; not faults of their own)
| concept | misconception | shadow_of | quote | score |
|---|---|---|---|---|
| for_loop | ForLoopPartSkipped | mis:loop_scope->ForVariableScopeBeyondLoop | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | 0.275 |
| loop_control | ForLoopPartSkipped | mis:loop_body->LoopBodyStatementOutsideLoop | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | 0.4 |

### Broken concept edges
| edge | quote | score | consequence_of |
|---|---|---|---|
| pre:for_loop->nested_loops | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | 0.644 | mis:loop_control->ForLoopPartSkipped |
| pre:loop_control->for_loop | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | 0.51 | mis:loop_control->ForLoopPartSkipped |
| pre:for_loop->loop_body | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | 0.17 | mis:loop_control->ForLoopPartSkipped |
| pre:for_loop->loop_scope | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | 0.06 | mis:loop_control->ForLoopPartSkipped |
| pre:statement->control_flow | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | 0.8 | mis:loop_control->ForLoopPartSkipped |
| pre:statement->println | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | 0.48 | mis:loop_control->ForLoopPartSkipped |

### Root cause: nested_loops / NestedLoopsSequential

### Level per concept
| concept | level | way | misconception | source | quote | rule | also | note |
|---|---|---|---|---|---|---|---|---|
| control_flow | L2 | output_prediction |  | level_way | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |  | L0 inconsistent_trace, L0 everyday_reading |  |
| loop_body | L0 | misconception | LoopBodyStatementOutsideLoop | level_mis_links | Three lines; The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | wrong rule: Some statements inside the braces of a loop body are not repeated; they run once, before or after the loop. Right rule: Every statement inside the loop braces is part of the body and runs on every pass. |  |  |
| loop_control | L2 | output_prediction |  | level_way | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |  |  |  |
| loop_scope | L0 | misconception | ForVariableScopeBeyondLoop | level_mis_links | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | wrong rule: i declared in the for header can still be used after the loop has ended. Right rule: i declared in the for header exists only inside the loop, so using it after the loop does not compile. |  |  |
| nested_loops | L0 | misconception | NestedLoopsSequential | level_mis_links | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | wrong rule: The outer loop runs completely, then the inner loop runs completely. Right rule: The inner loop runs all its passes during each single pass of the outer loop. |  |  |
| println | L3 | relation_to_neighbour |  | level_way | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |  | L3 purpose_of_the_whole, L2 output_prediction |  |
| statement | L3 | relation_to_neighbour |  | level_way | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |  |  |  |
| string_concatenation | L3 | purpose_of_the_whole |  | level_way | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |  | L3 relation_to_neighbour |  |

### Overall level: L0

Clause coverage (final):
| clause | covered by |
|---|---|
| Three lines | lvl:loop_control@L2#1 |
| The outer loop finishes first | mis:for_loop->ForLoopPartSkipped |
| so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 | mis:for_loop->ForLoopPartSkipped |

**Figure 8 (final picture)**

![Figure 8](figure08.png)


## 5. Every labelled edge (the evidence list, kept or not)
| depth | source | edge | kind | label | prune_score | confidence | score | clauses | quote |
|---|---|---|---|---|---|---|---|---|---|
| 1 | for_loop | mis:for_loop->ForLoopPartSkipped | misconception | held | 0.5 | 0.55 | 0.275 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 1 | loop_control | mis:loop_control->ForLoopPartSkipped | misconception | held | 0.5 | 0.8 | 0.4 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 1 | loop_control | lvl:loop_control@L2#1 | level | shown | 0.3 | 0.6 | 0.18 | [1, 3] | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 1 | string_concatenation | lvl:string_concatenation@L3#3 | level | shown | 0.3 | 0.6 | 0.18 | [3] | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 1 | string_concatenation | lvl:string_concatenation@L3#1 | level | shown | 0.25 | 0.6 | 0.15 | [3] | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 2 | loop_control | pre:loop_control->for_loop | prerequisite | broken | 0.6000000000000001 | 0.85 | 0.51 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 2 | loop_control | pre:loop_variable->loop_control | prerequisite | confirmed | 0.30000000000000004 | 0.6 | 0.18 | [3] | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 2 | for_loop | pre:for_loop->nested_loops | prerequisite | broken | 0.7000000000000001 | 0.92 | 0.644 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 2 | for_loop | pre:for_loop->loop_body | prerequisite | broken | 0.20000000000000004 | 0.85 | 0.17 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 2 | for_loop | pre:for_loop->loop_scope | prerequisite | broken | 0.10000000000000002 | 0.6 | 0.06 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 2 | string_concatenation | pre:string_concatenation->string_conversion | prerequisite | confirmed | 1.0 | 0.6 | 0.6 | [3] | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 2 | control_flow | mis:control_flow->ProgramEndsAfterLoop | misconception | correct | 0.15 | 0.55 | 0.0825 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 2 | control_flow | lvl:control_flow@L0#3 | level | shown | 0.5 | 0.7 | 0.35 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 2 | control_flow | lvl:control_flow@L0#0 | level | shown | 0.3 | 0.7 | 0.21 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 2 | control_flow | lvl:control_flow@L2#1 | level | shown | 0.2 | 0.6 | 0.12 | [1, 3] | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 2 | println | lvl:println@L2#1 | level | shown | 0.5 | 0.6 | 0.3 | [1, 3] | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 2 | println | lvl:println@L3#0 | level | shown | 0.3 | 0.6 | 0.18 | [3] | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 2 | println | lvl:println@L3#4 | level | shown | 0.2 | 0.6 | 0.12 | [1, 3] | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 3 | nested_loops | mis:nested_loops->NestedLoopsSequential | misconception | held | 0.6 | 0.95 | 0.57 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 3 | nested_loops | lvl:nested_loops@L0#7 | level | shown | 0.5 | 0.7 | 0.35 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 3 | nested_loops | lvl:nested_loops@L0#0 | level | shown | 0.3 | 0.9 | 0.27 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 3 | loop_control | pre:loop_variable->loop_control | prerequisite | confirmed | 0.8 | 0.6 | 0.48 | [3] | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 3 | loop_body | mis:loop_body->LoopBodyStatementOutsideLoop | misconception | held | 0.5 | 0.85 | 0.425 | [1, 2, 3] | Three lines; The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 3 | loop_body | lvl:loop_body@L0#0 | level | shown | 0.3 | 0.8 | 0.24 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 3 | loop_body | lvl:loop_body@L2#1 | level | shown | 0.25 | 0.7 | 0.175 | [1, 3] | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 3 | loop_scope | mis:loop_scope->ForVariableScopeBeyondLoop | misconception | held | 0.6 | 0.6 | 0.36 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 3 | loop_scope | lvl:loop_scope@L0#0 | level | shown | 0.45 | 0.6 | 0.27 | [3] | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 3 | control_flow | pre:statement->control_flow | prerequisite | broken | 1.0 | 0.8 | 0.8 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 3 | println | pre:statement->println | prerequisite | broken | 0.8 | 0.6 | 0.48 | [1, 3] | Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 3 | println | pre:string_literal->println | prerequisite | confirmed | 0.2 | 0.6 | 0.12 | [3] | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |
| 4 | statement | lvl:statement@L3#4 | level | shown | 0.25 | 0.6 | 0.15 | [2, 3] | The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3 |


## 6. The paths (what the keep step kept, depth by depth)
| depth | source | edge | kind | label | score | clauses |
|---|---|---|---|---|---|---|
| 1 | loop_control | mis:loop_control->ForLoopPartSkipped | misconception | held | 0.4 | [2, 3] |
| 1 | for_loop | mis:for_loop->ForLoopPartSkipped | misconception | held | 0.275 | [2, 3] |
| 1 | loop_control | lvl:loop_control@L2#1 | level | shown | 0.18 | [1, 3] |
| 2 | for_loop | pre:for_loop->nested_loops | prerequisite | broken | 0.644 | [2, 3] |
| 2 | loop_control | pre:loop_control->for_loop | prerequisite | broken | 0.51 | [2, 3] |
| 2 | for_loop | pre:for_loop->loop_body | prerequisite | broken | 0.17 | [2, 3] |
| 2 | for_loop | pre:for_loop->loop_scope | prerequisite | broken | 0.06 | [2, 3] |
| 3 | control_flow | pre:statement->control_flow | prerequisite | broken | 0.8 | [2, 3] |
| 3 | nested_loops | mis:nested_loops->NestedLoopsSequential | misconception | held | 0.57 | [2, 3] |
| 3 | println | pre:statement->println | prerequisite | broken | 0.48 | [1, 3] |
| 3 | loop_body | mis:loop_body->LoopBodyStatementOutsideLoop | misconception | held | 0.425 | [1, 2, 3] |
| 3 | loop_scope | mis:loop_scope->ForVariableScopeBeyondLoop | misconception | held | 0.36 | [2, 3] |
| 4 | statement | lvl:statement@L3#4 | level | shown | 0.15 | [2, 3] |


## 7. Raw traversal log
```
clauses: Three lines | The outer loop finishes first | so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3
start: BM25 of the answer against each tagged concept: for_loop 21.85 (hits 1), loop_control 20.94 (hits 1), string_concatenation 19.33 (hits 1), string_conversion 18.38 (hits 2), increment_operators 18.11 (hits 1), nested_loops 14.66 (hits 1), control_flow 11.57 (hits 1), loop_body 9.71 (hits 1), loop_variable 9.13 (hits 1), println 6.32 (hits 1), loop_scope 5.43 (hits 1), boolean_expression 4.59 (hits 1), comparison_operators 0.89 (hits 1)
start: chosen for_loop, loop_control, string_concatenation
start concepts (3): for_loop, loop_control, string_concatenation
--- depth 1: current concepts ['for_loop', 'loop_control', 'string_concatenation']
  for_loop (misconceptions): 21 candidates -> kept mis:for_loop->ForLoopPartSkipped (0.50), err:for_loop->WrongIterationCount (0.30), err:for_loop->BracelessMultiStatementBody (0.20)
    mis:for_loop->ForLoopPartSkipped -> held (0.28) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    err:for_loop->WrongIterationCount -> not applied (0.00) ""
    err:for_loop->BracelessMultiStatementBody -> not applied (0.00) ""
  for_loop (level ways): 24 candidates -> kept lvl:for_loop@L3#1 (0.60), lvl:for_loop@L0#11 (0.20), lvl:for_loop@L2#1 (0.20)
    lvl:for_loop@L3#1 -> not applied (0.00) ""
    lvl:for_loop@L0#11 -> not applied (0.00) ""
    lvl:for_loop@L2#1 -> not applied (0.00) ""
  loop_control (misconceptions): 11 candidates -> kept mis:loop_control->ForLoopPartSkipped (0.50), err:loop_control->WrongIterationCount (0.30), mis:loop_control->IdiomOverridesCode (0.20)
    mis:loop_control->ForLoopPartSkipped -> held (0.40) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    err:loop_control->WrongIterationCount -> not applied (0.00) ""
    mis:loop_control->IdiomOverridesCode -> not applied (0.00) ""
  loop_control (level ways): 19 candidates -> kept lvl:loop_control@L0#3 (0.50), lvl:loop_control@L2#1 (0.30), lvl:loop_control@L2#0 (0.20)
    lvl:loop_control@L0#3 -> not applied (0.00) ""
    lvl:loop_control@L2#1 -> shown (0.18) "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    lvl:loop_control@L2#0 -> not applied (0.00) ""
  string_concatenation (misconceptions): 3 candidates -> kept mis:string_concatenation->ArithmeticPlusPrecedes (0.60), mis:string_concatenation->StringPlusLeftToRight (0.30), mis:string_concatenation->StringPlusStringifiesExpression (0.10)
    mis:string_concatenation->ArithmeticPlusPrecedes -> not applied (0.00) ""
    mis:string_concatenation->StringPlusLeftToRight -> not applied (0.00) ""
    mis:string_concatenation->StringPlusStringifiesExpression -> not applied (0.00) ""
  string_concatenation (level ways): 16 candidates -> kept lvl:string_concatenation@L2#1 (0.45), lvl:string_concatenation@L3#3 (0.30), lvl:string_concatenation@L3#1 (0.25)
    lvl:string_concatenation@L2#1 -> not applied (0.00) ""
    lvl:string_concatenation@L3#3 -> shown (0.18) "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    lvl:string_concatenation@L3#1 -> shown (0.15) "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  kept: mis:loop_control->ForLoopPartSkipped [held, tier 0, 0.40], mis:for_loop->ForLoopPartSkipped [held, tier 0, 0.28], lvl:loop_control@L2#1 [shown, tier 2, 0.18]
  clause coverage: c1 uncovered; c2 <- mis:for_loop->ForLoopPartSkipped; c3 <- mis:for_loop->ForLoopPartSkipped
  uncovered clauses bring in: control_flow (1.84), println (1.53)
  not yet read: own edges of control_flow, own edges of println, pre:for_loop->loop_body, pre:for_loop->loop_scope, pre:for_loop->nested_loops, pre:loop_control->for_loop, pre:loop_control->increment_operators, pre:loop_variable->loop_control, pre:string_concatenation->string_conversion
--- depth 2: current concepts ['loop_control', 'for_loop', 'string_concatenation', 'control_flow', 'println']
  loop_control (concept edges): 3 candidates -> kept pre:loop_control->for_loop (0.60), pre:loop_variable->loop_control (0.30), pre:loop_control->increment_operators (0.10)
    pre:loop_control->for_loop -> broken (0.51) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    pre:loop_variable->loop_control -> confirmed (0.18) "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    pre:loop_control->increment_operators -> not applied (0.00) ""
  for_loop (concept edges): 3 candidates -> kept pre:for_loop->nested_loops (0.70), pre:for_loop->loop_body (0.20), pre:for_loop->loop_scope (0.10)
    pre:for_loop->nested_loops -> broken (0.64) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    pre:for_loop->loop_body -> broken (0.17) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    pre:for_loop->loop_scope -> broken (0.06) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  string_concatenation (concept edges): 1 candidates -> kept pre:string_concatenation->string_conversion (1.00)
    pre:string_concatenation->string_conversion -> confirmed (0.60) "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  control_flow (misconceptions): 11 candidates -> kept mis:control_flow->StatementsRunInAnyOrder (0.60), mis:control_flow->FalseLoopRepeatsNextCode (0.25), mis:control_flow->ProgramEndsAfterLoop (0.15)
    mis:control_flow->StatementsRunInAnyOrder -> not applied (0.00) ""
    mis:control_flow->FalseLoopRepeatsNextCode -> not applied (0.00) ""
    mis:control_flow->ProgramEndsAfterLoop -> correct (0.08) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  control_flow (level ways): 15 candidates -> kept lvl:control_flow@L0#3 (0.50), lvl:control_flow@L0#0 (0.30), lvl:control_flow@L2#1 (0.20)
    lvl:control_flow@L0#3 -> shown (0.35) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    lvl:control_flow@L0#0 -> shown (0.21) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    lvl:control_flow@L2#1 -> shown (0.12) "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  println (misconceptions): 6 candidates -> kept mis:println->PrintEqualsPrintln (0.60), mis:println->PrintNewLineFirst (0.30), mis:println->PrintedTextIsExecuted (0.10)
    mis:println->PrintEqualsPrintln -> not applied (0.00) ""
    mis:println->PrintNewLineFirst -> not applied (0.00) ""
    mis:println->PrintedTextIsExecuted -> not applied (0.00) ""
  println (level ways): 17 candidates -> kept lvl:println@L2#1 (0.50), lvl:println@L3#0 (0.30), lvl:println@L3#4 (0.20)
    lvl:println@L2#1 -> shown (0.30) "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    lvl:println@L3#0 -> shown (0.18) "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    lvl:println@L3#4 -> shown (0.12) "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  kept: pre:for_loop->nested_loops [broken, tier 0, 0.64], pre:loop_control->for_loop [broken, tier 0, 0.51], pre:for_loop->loop_body [broken, tier 0, 0.17], pre:for_loop->loop_scope [broken, tier 0, 0.06]
  reached across pre:for_loop->nested_loops, entered next: nested_loops
  reached across pre:for_loop->loop_body, entered next: loop_body
  reached across pre:for_loop->loop_scope, entered next: loop_scope
  clause coverage: c1 uncovered; c2 <- mis:for_loop->ForLoopPartSkipped; c3 <- mis:for_loop->ForLoopPartSkipped
  not yet read: own edges of loop_body, own edges of loop_scope, own edges of nested_loops, pre:statement->control_flow, pre:statement->println, pre:string_literal->println
--- depth 3: current concepts ['nested_loops', 'for_loop', 'loop_control', 'loop_body', 'loop_scope', 'control_flow', 'println']
  nested_loops (misconceptions): 8 candidates -> kept mis:nested_loops->NestedLoopsSequential (0.60), mis:nested_loops->InnerCounterOverwritesOuter (0.25), mis:nested_loops->OuterLoopIgnored (0.15)
    mis:nested_loops->NestedLoopsSequential -> held (0.57) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    mis:nested_loops->InnerCounterOverwritesOuter -> not applied (0.00) ""
    mis:nested_loops->OuterLoopIgnored -> not applied (0.00) ""
  nested_loops (level ways): 17 candidates -> kept lvl:nested_loops@L0#7 (0.50), lvl:nested_loops@L0#0 (0.30), lvl:nested_loops@L2#1 (0.20)
    lvl:nested_loops@L0#7 -> shown (0.35) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    lvl:nested_loops@L0#0 -> shown (0.27) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    lvl:nested_loops@L2#1 -> not applied (0.00) ""
  loop_control (concept edges): 2 candidates -> kept pre:loop_variable->loop_control (0.80), pre:loop_control->increment_operators (0.20)
    pre:loop_variable->loop_control -> confirmed (0.48) "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    pre:loop_control->increment_operators -> not applied (0.00) ""
  loop_body (misconceptions): 12 candidates -> kept mis:loop_body->LoopBodyStatementOutsideLoop (0.50), mis:loop_body->ForLoopPartSkipped (0.30), mis:loop_body->LoopRunsStatementsSeparately (0.20)
    mis:loop_body->LoopBodyStatementOutsideLoop -> held (0.42) "Three lines; The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    mis:loop_body->ForLoopPartSkipped -> not applied (0.00) ""
    mis:loop_body->LoopRunsStatementsSeparately -> not applied (0.00) ""
  loop_body (level ways): 14 candidates -> kept lvl:loop_body@L0#6 (0.45), lvl:loop_body@L0#0 (0.30), lvl:loop_body@L2#1 (0.25)
    lvl:loop_body@L0#6 -> not applied (0.00) ""
    lvl:loop_body@L0#0 -> shown (0.24) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    lvl:loop_body@L2#1 -> shown (0.17) "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  loop_scope (misconceptions): 5 candidates -> kept mis:loop_scope->ForVariableScopeBeyondLoop (0.60), mis:loop_scope->LocalVarOutsideScope (0.25), mis:loop_scope->LoopVariableLifetimeScope (0.15)
    mis:loop_scope->ForVariableScopeBeyondLoop -> held (0.36) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    mis:loop_scope->LocalVarOutsideScope -> not applied (0.00) ""
    mis:loop_scope->LoopVariableLifetimeScope -> not applied (0.00) ""
  loop_scope (level ways): 14 candidates -> kept lvl:loop_scope@L0#0 (0.45), lvl:loop_scope@L4#3 (0.35), lvl:loop_scope@L0#5 (0.20)
    lvl:loop_scope@L0#0 -> shown (0.27) "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    lvl:loop_scope@L4#3 -> not applied (0.00) ""
    lvl:loop_scope@L0#5 -> not applied (0.00) ""
  control_flow (concept edges): 1 candidates -> kept pre:statement->control_flow (1.00)
    pre:statement->control_flow -> broken (0.80) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  println (concept edges): 2 candidates -> kept pre:statement->println (0.80), pre:string_literal->println (0.20)
    pre:statement->println -> broken (0.48) "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
    pre:string_literal->println -> confirmed (0.12) "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  kept: pre:statement->control_flow [broken, tier 0, 0.80], mis:nested_loops->NestedLoopsSequential [held, tier 0, 0.57], pre:statement->println [broken, tier 0, 0.48], mis:loop_body->LoopBodyStatementOutsideLoop [held, tier 0, 0.42], mis:loop_scope->ForVariableScopeBeyondLoop [held, tier 0, 0.36]
  reached across pre:statement->control_flow, entered next: statement
  clause coverage: c1 <- mis:loop_body->LoopBodyStatementOutsideLoop; c2 <- mis:for_loop->ForLoopPartSkipped; c3 <- mis:for_loop->ForLoopPartSkipped
  not yet read: own edges of statement
--- depth 4: current concepts ['statement'] (closing round: own edges only)
  statement (misconceptions): 5 candidates -> kept mis:statement->StatementsRunInAnyOrder (0.70), mis:statement->ProgramLooksAhead (0.20), mis:statement->WhitespaceIsSignificant (0.10)
    mis:statement->StatementsRunInAnyOrder -> not applied (0.00) ""
    mis:statement->ProgramLooksAhead -> not applied (0.00) ""
    mis:statement->WhitespaceIsSignificant -> not applied (0.00) ""
  statement (level ways): 19 candidates -> kept lvl:statement@L0#2 (0.50), lvl:statement@L0#0 (0.25), lvl:statement@L3#4 (0.25)
    lvl:statement@L0#2 -> not applied (0.00) ""
    lvl:statement@L0#0 -> not applied (0.00) ""
    lvl:statement@L3#4 -> shown (0.15) "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  kept: lvl:statement@L3#4 [shown, tier 2, 0.15]
  clause coverage: c1 <- mis:loop_body->LoopBodyStatementOutsideLoop; c2 <- mis:for_loop->ForLoopPartSkipped; c3 <- mis:for_loop->ForLoopPartSkipped
  held: mis:nested_loops->NestedLoopsSequential, mis:loop_body->LoopBodyStatementOutsideLoop, mis:loop_scope->ForVariableScopeBeyondLoop, mis:for_loop->ForLoopPartSkipped; filed at the dependent concept: mis:loop_control->ForLoopPartSkipped
  shadows: mis:for_loop->ForLoopPartSkipped (shadow of mis:loop_scope->ForVariableScopeBeyondLoop), mis:loop_control->ForLoopPartSkipped (shadow of mis:loop_body->LoopBodyStatementOutsideLoop)
  root-cause walk: held on ['loop_body', 'loop_scope', 'nested_loops'], broken at [] (consequences: pre:for_loop->nested_loops, pre:loop_control->for_loop, pre:for_loop->loop_body, pre:for_loop->loop_scope, pre:statement->control_flow, pre:statement->println), prerequisites {'increment_operators': ['assignment', 'loop_control'], 'boolean_expression': ['comparison_operators'], 'comparison_operators': ['expression'], 'loop_body': ['for_loop'], 'loop_scope': ['for_loop'], 'nested_loops': ['for_loop'], 'for_loop': ['loop_control'], 'loop_control': ['loop_variable'], 'control_flow': ['statement'], 'println': ['statement', 'string_literal'], 'string_conversion': ['string_concatenation']}
  level control_flow: L2 via output_prediction "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  level loop_body: L0 via misconception "Three lines; The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  level loop_control: L2 via output_prediction "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  level loop_scope: L0 via misconception "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  level nested_loops: L0 via misconception "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  level println: L3 via relation_to_neighbour "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  level statement: L3 via relation_to_neighbour "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
  level string_concatenation: L3 via purpose_of_the_whole "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
```


## 8. Full diagnosis JSON
```json
{
 "scenario": "MY11",
 "settings": {
  "width": 3,
  "depth": 3,
  "prune_tools": "llm",
  "model": "claude-opus-5-5",
  "backend": "claude-cli"
 },
 "clauses": [
  "Three lines",
  "The outer loop finishes first",
  "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3"
 ],
 "start_concepts": [
  "for_loop",
  "loop_control",
  "string_concatenation"
 ],
 "held_misconceptions": [
  {
   "concept": "nested_loops",
   "misconception": "NestedLoopsSequential",
   "kind": "misconception",
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "score": 0.57
  },
  {
   "concept": "loop_body",
   "misconception": "LoopBodyStatementOutsideLoop",
   "kind": "misconception",
   "quote": "Three lines; The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "score": 0.425
  },
  {
   "concept": "loop_scope",
   "misconception": "ForVariableScopeBeyondLoop",
   "kind": "misconception",
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "score": 0.36
  }
 ],
 "error_evidence": [],
 "shadowed": [
  {
   "concept": "for_loop",
   "misconception": "ForLoopPartSkipped",
   "shadow_of": "mis:loop_scope->ForVariableScopeBeyondLoop",
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "score": 0.275
  },
  {
   "concept": "loop_control",
   "misconception": "ForLoopPartSkipped",
   "shadow_of": "mis:loop_body->LoopBodyStatementOutsideLoop",
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "score": 0.4
  }
 ],
 "broken_edges": [
  {
   "edge": "pre:for_loop->nested_loops",
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "score": 0.644,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "edge": "pre:loop_control->for_loop",
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "score": 0.51,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "edge": "pre:for_loop->loop_body",
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "score": 0.17,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "edge": "pre:for_loop->loop_scope",
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "score": 0.06,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "edge": "pre:statement->control_flow",
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "score": 0.8,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "edge": "pre:statement->println",
   "quote": "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "score": 0.48,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  }
 ],
 "root_cause": {
  "concept": "nested_loops",
  "misconception": "NestedLoopsSequential"
 },
 "levels": {
  "control_flow": {
   "level": "L2",
   "way": "output_prediction",
   "quote": "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "edge": "lvl:control_flow@L2#1",
   "source": "level_way",
   "also_shown": [
    "L0 inconsistent_trace",
    "L0 everyday_reading"
   ]
  },
  "loop_body": {
   "level": "L0",
   "way": "misconception",
   "misconception": "LoopBodyStatementOutsideLoop",
   "quote": "Three lines; The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "edge": "mis:loop_body->LoopBodyStatementOutsideLoop",
   "rule": "wrong rule: Some statements inside the braces of a loop body are not repeated; they run once, before or after the loop. Right rule: Every statement inside the loop braces is part of the body and runs on every pass.",
   "source": "level_mis_links"
  },
  "loop_control": {
   "level": "L2",
   "way": "output_prediction",
   "quote": "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "edge": "lvl:loop_control@L2#1",
   "source": "level_way",
   "also_shown": []
  },
  "loop_scope": {
   "level": "L0",
   "way": "misconception",
   "misconception": "ForVariableScopeBeyondLoop",
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "edge": "mis:loop_scope->ForVariableScopeBeyondLoop",
   "rule": "wrong rule: i declared in the for header can still be used after the loop has ended. Right rule: i declared in the for header exists only inside the loop, so using it after the loop does not compile.",
   "source": "level_mis_links"
  },
  "nested_loops": {
   "level": "L0",
   "way": "misconception",
   "misconception": "NestedLoopsSequential",
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "edge": "mis:nested_loops->NestedLoopsSequential",
   "rule": "wrong rule: The outer loop runs completely, then the inner loop runs completely. Right rule: The inner loop runs all its passes during each single pass of the outer loop.",
   "source": "level_mis_links"
  },
  "println": {
   "level": "L3",
   "way": "relation_to_neighbour",
   "quote": "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "edge": "lvl:println@L3#0",
   "source": "level_way",
   "also_shown": [
    "L3 purpose_of_the_whole",
    "L2 output_prediction"
   ]
  },
  "statement": {
   "level": "L3",
   "way": "relation_to_neighbour",
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "edge": "lvl:statement@L3#4",
   "source": "level_way",
   "also_shown": []
  },
  "string_concatenation": {
   "level": "L3",
   "way": "purpose_of_the_whole",
   "quote": "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "edge": "lvl:string_concatenation@L3#3",
   "source": "level_way",
   "also_shown": [
    "L3 relation_to_neighbour"
   ]
  }
 },
 "overall_level": "L0",
 "paths": [
  {
   "depth": 1,
   "source": "loop_control",
   "edge": "mis:loop_control->ForLoopPartSkipped",
   "target": "ForLoopPartSkipped",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.8,
   "prune_score": 0.5,
   "score": 0.4,
   "logic": "A pass of a loop may leave out one of its parts (the condition test, the body, the update, or the initialization).",
   "twin_logic": "Each pass of a for loop runs test, then body, then update, every time; the initialization runs once, at the start.",
   "role": "context"
  },
  {
   "depth": 1,
   "source": "for_loop",
   "edge": "mis:for_loop->ForLoopPartSkipped",
   "target": "ForLoopPartSkipped",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.55,
   "prune_score": 0.5,
   "score": 0.275,
   "logic": "A pass of a loop may leave out one of its parts (the condition test, the body, the update, or the initialization).",
   "twin_logic": "Each pass of a for loop runs test, then body, then update, every time; the initialization runs once, at the start.",
   "role": "primary"
  },
  {
   "depth": 1,
   "source": "loop_control",
   "edge": "lvl:loop_control@L2#1",
   "target": "loop_control@L2",
   "kind": "level",
   "label": "shown",
   "clauses": [
    1,
    3
   ],
   "quote": "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.3,
   "score": 0.18,
   "logic": "The number of passes and the values seen follow from the three parts: for (int i = 0; i <= N; i++) makes N+1 passes with i = 0..N; with i < N it makes N passes; i += 2 visits every second value.",
   "twin_logic": null,
   "level": "L2",
   "way": "output_prediction"
  },
  {
   "depth": 2,
   "source": "for_loop",
   "edge": "pre:for_loop->nested_loops",
   "target": "nested_loops",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.92,
   "prune_score": 0.7000000000000001,
   "score": 0.644,
   "logic": "In nested loops the inner for runs its complete set of passes for every single pass of the outer for.",
   "twin_logic": null,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "depth": 2,
   "source": "loop_control",
   "edge": "pre:loop_control->for_loop",
   "target": "for_loop",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.85,
   "prune_score": 0.6000000000000001,
   "score": 0.51,
   "logic": "The for header is exactly the three control slots, and the body runs after the test and before the update on each pass.",
   "twin_logic": null,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "depth": 2,
   "source": "for_loop",
   "edge": "pre:for_loop->loop_body",
   "target": "loop_body",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.85,
   "prune_score": 0.20000000000000004,
   "score": 0.17,
   "logic": "The statements in braces after the for header are the loop body; they run once on every pass, between the test and the increment, exactly as in a while loop.",
   "twin_logic": null,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "depth": 2,
   "source": "for_loop",
   "edge": "pre:for_loop->loop_scope",
   "target": "loop_scope",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.10000000000000002,
   "score": 0.06,
   "logic": "A variable declared in the for header exists only from its declaration to the end of the for statement's block.",
   "twin_logic": null,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "depth": 3,
   "source": "control_flow",
   "edge": "pre:statement->control_flow",
   "target": "statement",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.8,
   "prune_score": 1.0,
   "score": 0.8,
   "logic": "Control flow is the order in which statements run.",
   "twin_logic": null,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "depth": 3,
   "source": "nested_loops",
   "edge": "mis:nested_loops->NestedLoopsSequential",
   "target": "NestedLoopsSequential",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.95,
   "prune_score": 0.6,
   "score": 0.57,
   "logic": "The outer loop runs completely, then the inner loop runs completely.",
   "twin_logic": "The inner loop runs all its passes during each single pass of the outer loop.",
   "role": "primary"
  },
  {
   "depth": 3,
   "source": "println",
   "edge": "pre:statement->println",
   "target": "statement",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    1,
    3
   ],
   "quote": "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.8,
   "score": 0.48,
   "logic": "System.out.println(s); is itself a statement, so its line of output appears at its own place in the sequence.",
   "twin_logic": null,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "depth": 3,
   "source": "loop_body",
   "edge": "mis:loop_body->LoopBodyStatementOutsideLoop",
   "target": "LoopBodyStatementOutsideLoop",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "Three lines; The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.85,
   "prune_score": 0.5,
   "score": 0.425,
   "logic": "Some statements inside the braces of a loop body are not repeated; they run once, before or after the loop.",
   "twin_logic": "Every statement inside the loop braces is part of the body and runs on every pass.",
   "role": "primary"
  },
  {
   "depth": 3,
   "source": "loop_scope",
   "edge": "mis:loop_scope->ForVariableScopeBeyondLoop",
   "target": "ForVariableScopeBeyondLoop",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.6,
   "score": 0.36,
   "logic": "i declared in the for header can still be used after the loop has ended.",
   "twin_logic": "i declared in the for header exists only inside the loop, so using it after the loop does not compile.",
   "role": "primary"
  },
  {
   "depth": 4,
   "source": "statement",
   "edge": "lvl:statement@L3#4",
   "target": "statement@L3",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.25,
   "score": 0.15,
   "logic": "Control flow is the order in which statements run; without control statements it is straight-line, each once in order.",
   "twin_logic": null,
   "level": "L3",
   "way": "relation_to_neighbour"
  }
 ],
 "evidence": [
  {
   "depth": 1,
   "source": "for_loop",
   "edge": "mis:for_loop->ForLoopPartSkipped",
   "target": "ForLoopPartSkipped",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.55,
   "prune_score": 0.5,
   "score": 0.275,
   "logic": "A pass of a loop may leave out one of its parts (the condition test, the body, the update, or the initialization).",
   "twin_logic": "Each pass of a for loop runs test, then body, then update, every time; the initialization runs once, at the start.",
   "role": "primary"
  },
  {
   "depth": 1,
   "source": "loop_control",
   "edge": "mis:loop_control->ForLoopPartSkipped",
   "target": "ForLoopPartSkipped",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.8,
   "prune_score": 0.5,
   "score": 0.4,
   "logic": "A pass of a loop may leave out one of its parts (the condition test, the body, the update, or the initialization).",
   "twin_logic": "Each pass of a for loop runs test, then body, then update, every time; the initialization runs once, at the start.",
   "role": "context"
  },
  {
   "depth": 1,
   "source": "loop_control",
   "edge": "lvl:loop_control@L2#1",
   "target": "loop_control@L2",
   "kind": "level",
   "label": "shown",
   "clauses": [
    1,
    3
   ],
   "quote": "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.3,
   "score": 0.18,
   "logic": "The number of passes and the values seen follow from the three parts: for (int i = 0; i <= N; i++) makes N+1 passes with i = 0..N; with i < N it makes N passes; i += 2 visits every second value.",
   "twin_logic": null,
   "level": "L2",
   "way": "output_prediction"
  },
  {
   "depth": 1,
   "source": "string_concatenation",
   "edge": "lvl:string_concatenation@L3#3",
   "target": "string_concatenation@L3",
   "kind": "level",
   "label": "shown",
   "clauses": [
    3
   ],
   "quote": "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.3,
   "score": 0.18,
   "logic": "Concatenation builds output lines and labels from computed values (a + \" + \" + b + \" = \" + c). Repeated s = s + piece makes s a gatherer of text.",
   "twin_logic": null,
   "level": "L3",
   "way": "purpose_of_the_whole"
  },
  {
   "depth": 1,
   "source": "string_concatenation",
   "edge": "lvl:string_concatenation@L3#1",
   "target": "string_concatenation@L3",
   "kind": "level",
   "label": "shown",
   "clauses": [
    3
   ],
   "quote": "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.25,
   "score": 0.15,
   "logic": "The non-String operand is converted to its printed form (int 99 to \"99\", double 1.0 to \"1.0\", char 'a' to \"a\") before joining.",
   "twin_logic": null,
   "level": "L3",
   "way": "relation_to_neighbour"
  },
  {
   "depth": 2,
   "source": "loop_control",
   "edge": "pre:loop_control->for_loop",
   "target": "for_loop",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.85,
   "prune_score": 0.6000000000000001,
   "score": 0.51,
   "logic": "The for header is exactly the three control slots, and the body runs after the test and before the update on each pass.",
   "twin_logic": null,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "depth": 2,
   "source": "loop_control",
   "edge": "pre:loop_variable->loop_control",
   "target": "loop_variable",
   "kind": "prerequisite",
   "label": "confirmed",
   "clauses": [
    3
   ],
   "quote": "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.30000000000000004,
   "score": 0.18,
   "logic": "The init, condition and update all act on the loop variable: the init sets it, the condition tests it, the update changes it.",
   "twin_logic": null
  },
  {
   "depth": 2,
   "source": "for_loop",
   "edge": "pre:for_loop->nested_loops",
   "target": "nested_loops",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.92,
   "prune_score": 0.7000000000000001,
   "score": 0.644,
   "logic": "In nested loops the inner for runs its complete set of passes for every single pass of the outer for.",
   "twin_logic": null,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "depth": 2,
   "source": "for_loop",
   "edge": "pre:for_loop->loop_body",
   "target": "loop_body",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.85,
   "prune_score": 0.20000000000000004,
   "score": 0.17,
   "logic": "The statements in braces after the for header are the loop body; they run once on every pass, between the test and the increment, exactly as in a while loop.",
   "twin_logic": null,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "depth": 2,
   "source": "for_loop",
   "edge": "pre:for_loop->loop_scope",
   "target": "loop_scope",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.10000000000000002,
   "score": 0.06,
   "logic": "A variable declared in the for header exists only from its declaration to the end of the for statement's block.",
   "twin_logic": null,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "depth": 2,
   "source": "string_concatenation",
   "edge": "pre:string_concatenation->string_conversion",
   "target": "string_conversion",
   "kind": "prerequisite",
   "label": "confirmed",
   "clauses": [
    3
   ],
   "quote": "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 1.0,
   "score": 0.6,
   "logic": "When one operand of + is a String, Java converts the other operand to a String before concatenating.",
   "twin_logic": null
  },
  {
   "depth": 2,
   "source": "control_flow",
   "edge": "mis:control_flow->ProgramEndsAfterLoop",
   "target": "ProgramEndsAfterLoop",
   "kind": "misconception",
   "label": "correct",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.55,
   "prune_score": 0.15,
   "score": 0.0825,
   "logic": "When a loop finishes the program ends, so code after the loop never runs.",
   "twin_logic": "When the loop condition becomes false, control continues with the first statement after the loop.",
   "role": "context"
  },
  {
   "depth": 2,
   "source": "control_flow",
   "edge": "lvl:control_flow@L0#3",
   "target": "control_flow@L0",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.7,
   "prune_score": 0.5,
   "score": 0.35,
   "logic": "Applies the sequencing rule (LT) unevenly: e.g. runs a later line before an earlier one, lets an if that was already passed fire again later, or ends the program when a loop or a false if ends instead of continuing with the next statement. (here mixing in: \"Statements in a program can run in any order\")",
   "twin_logic": null,
   "level": "L0",
   "way": "inconsistent_trace"
  },
  {
   "depth": 2,
   "source": "control_flow",
   "edge": "lvl:control_flow@L0#0",
   "target": "control_flow@L0",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.7,
   "prune_score": 0.3,
   "score": 0.21,
   "logic": "Folk meaning: in everyday instructions a listener reads the whole list at once, does steps in a sensible order and infers what the writer wants. Java instead executes statements one at a time, top to bottom, each only when control reaches it, and nothing later in the program has any effect earlier.",
   "twin_logic": null,
   "level": "L0",
   "way": "everyday_reading"
  },
  {
   "depth": 2,
   "source": "control_flow",
   "edge": "lvl:control_flow@L2#1",
   "target": "control_flow@L2",
   "kind": "level",
   "label": "shown",
   "clauses": [
    1,
    3
   ],
   "quote": "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.2,
   "score": 0.12,
   "logic": "The output is the sequence of print statements that control actually reaches, in the order reached: e.g. TenHellos prints 1st, 2nd, 3rd Hello once each, then 4th to 10th Hello from the loop.",
   "twin_logic": null,
   "level": "L2",
   "way": "output_prediction"
  },
  {
   "depth": 2,
   "source": "println",
   "edge": "lvl:println@L2#1",
   "target": "println@L2",
   "kind": "level",
   "label": "shown",
   "clauses": [
    1,
    3
   ],
   "quote": "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.5,
   "score": 0.3,
   "logic": "Output is the value's characters followed by a line break: println(\"Hello, \"); println(\"World!\"); prints two lines.",
   "twin_logic": null,
   "level": "L2",
   "way": "output_prediction"
  },
  {
   "depth": 2,
   "source": "println",
   "edge": "lvl:println@L3#0",
   "target": "println@L3",
   "kind": "level",
   "label": "shown",
   "clauses": [
    3
   ],
   "quote": "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.3,
   "score": 0.18,
   "logic": "System.out.println(s); is a statement, so its line of output appears at its own place in the sequence.",
   "twin_logic": null,
   "level": "L3",
   "way": "relation_to_neighbour"
  },
  {
   "depth": 2,
   "source": "println",
   "edge": "lvl:println@L3#4",
   "target": "println@L3",
   "kind": "level",
   "label": "shown",
   "clauses": [
    1,
    3
   ],
   "quote": "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.2,
   "score": 0.12,
   "logic": "Purpose: produce one complete line of output, e.g. each result on its own line, or end a line built with print.",
   "twin_logic": null,
   "level": "L3",
   "way": "purpose_of_the_whole"
  },
  {
   "depth": 3,
   "source": "nested_loops",
   "edge": "mis:nested_loops->NestedLoopsSequential",
   "target": "NestedLoopsSequential",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.95,
   "prune_score": 0.6,
   "score": 0.57,
   "logic": "The outer loop runs completely, then the inner loop runs completely.",
   "twin_logic": "The inner loop runs all its passes during each single pass of the outer loop.",
   "role": "primary"
  },
  {
   "depth": 3,
   "source": "nested_loops",
   "edge": "lvl:nested_loops@L0#7",
   "target": "nested_loops@L0",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.7,
   "prune_score": 0.5,
   "score": 0.35,
   "logic": "Applies the nested-loop rule (LT) unevenly: resets the inner variable on some outer passes but not others, or advances the outer variable in the middle of the inner loop. (here mixing in: \"The outer loop runs completely, then the inner loop runs completely\")",
   "twin_logic": null,
   "level": "L0",
   "way": "inconsistent_trace"
  },
  {
   "depth": 3,
   "source": "nested_loops",
   "edge": "lvl:nested_loops@L0#0",
   "target": "nested_loops@L0",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.9,
   "prune_score": 0.3,
   "score": 0.27,
   "logic": "Folk meaning: 'for each row and each column' sounds like doing two things side by side or one after the other. In Java the inner loop is a statement in the outer body, so it runs all its passes during each single outer pass (like the hands of a clock).",
   "twin_logic": null,
   "level": "L0",
   "way": "everyday_reading"
  },
  {
   "depth": 3,
   "source": "loop_control",
   "edge": "pre:loop_variable->loop_control",
   "target": "loop_variable",
   "kind": "prerequisite",
   "label": "confirmed",
   "clauses": [
    3
   ],
   "quote": "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.8,
   "score": 0.48,
   "logic": "The init, condition and update all act on the loop variable: the init sets it, the condition tests it, the update changes it.",
   "twin_logic": null
  },
  {
   "depth": 3,
   "source": "loop_body",
   "edge": "mis:loop_body->LoopBodyStatementOutsideLoop",
   "target": "LoopBodyStatementOutsideLoop",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "Three lines; The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.85,
   "prune_score": 0.5,
   "score": 0.425,
   "logic": "Some statements inside the braces of a loop body are not repeated; they run once, before or after the loop.",
   "twin_logic": "Every statement inside the loop braces is part of the body and runs on every pass.",
   "role": "primary"
  },
  {
   "depth": 3,
   "source": "loop_body",
   "edge": "lvl:loop_body@L0#0",
   "target": "loop_body@L0",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.8,
   "prune_score": 0.3,
   "score": 0.24,
   "logic": "Folk meaning: an everyday 'repeat this' instruction can be read loosely - a helper fills in missing steps, does each step N times in turn, or stops partway. A Java loop body is exactly the statements inside its braces (or one statement), all executed in order on every pass.",
   "twin_logic": null,
   "level": "L0",
   "way": "everyday_reading"
  },
  {
   "depth": 3,
   "source": "loop_body",
   "edge": "lvl:loop_body@L2#1",
   "target": "loop_body@L2",
   "kind": "level",
   "label": "shown",
   "clauses": [
    1,
    3
   ],
   "quote": "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.7,
   "prune_score": 0.25,
   "score": 0.175,
   "logic": "The body's output on each pass depends on the current values, so successive passes can print different lines: TenHellos prints '4th Hello', '5th Hello', ...; the number of executions of each body statement equals the number of passes.",
   "twin_logic": null,
   "level": "L2",
   "way": "output_prediction"
  },
  {
   "depth": 3,
   "source": "loop_scope",
   "edge": "mis:loop_scope->ForVariableScopeBeyondLoop",
   "target": "ForVariableScopeBeyondLoop",
   "kind": "misconception",
   "label": "held",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.6,
   "score": 0.36,
   "logic": "i declared in the for header can still be used after the loop has ended.",
   "twin_logic": "i declared in the for header exists only inside the loop, so using it after the loop does not compile.",
   "role": "primary"
  },
  {
   "depth": 3,
   "source": "loop_scope",
   "edge": "lvl:loop_scope@L0#0",
   "target": "loop_scope@L0",
   "kind": "level",
   "label": "shown",
   "clauses": [
    3
   ],
   "quote": "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.45,
   "score": 0.27,
   "logic": "Folk meaning: once a name is introduced ('let i be ...') it can be referred to for the rest of the text, and something introduced inside a repeated step keeps existing across repetitions. In Java a local variable exists only from its declaration to the end of its block, and a for header counts as part of the loop's block.",
   "twin_logic": null,
   "level": "L0",
   "way": "everyday_reading"
  },
  {
   "depth": 3,
   "source": "control_flow",
   "edge": "pre:statement->control_flow",
   "target": "statement",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.8,
   "prune_score": 1.0,
   "score": 0.8,
   "logic": "Control flow is the order in which statements run.",
   "twin_logic": null,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "depth": 3,
   "source": "println",
   "edge": "pre:statement->println",
   "target": "statement",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    1,
    3
   ],
   "quote": "Three lines; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.8,
   "score": 0.48,
   "logic": "System.out.println(s); is itself a statement, so its line of output appears at its own place in the sequence.",
   "twin_logic": null,
   "consequence_of": "mis:loop_control->ForLoopPartSkipped"
  },
  {
   "depth": 3,
   "source": "println",
   "edge": "pre:string_literal->println",
   "target": "string_literal",
   "kind": "prerequisite",
   "label": "confirmed",
   "clauses": [
    3
   ],
   "quote": "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.2,
   "score": 0.12,
   "logic": "println(\"s\") outputs exactly the characters between the quotes and then ends the line.",
   "twin_logic": null
  },
  {
   "depth": 4,
   "source": "statement",
   "edge": "lvl:statement@L3#4",
   "target": "statement@L3",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The outer loop finishes first; so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3",
   "confidence": 0.6,
   "prune_score": 0.25,
   "score": 0.15,
   "logic": "Control flow is the order in which statements run; without control statements it is straight-line, each once in order.",
   "twin_logic": null,
   "level": "L3",
   "way": "relation_to_neighbour"
  }
 ],
 "clause_coverage": {
  "Three lines": "lvl:loop_control@L2#1",
  "The outer loop finishes first": "mis:for_loop->ForLoopPartSkipped",
  "so i ends at 2, then the inner loop runs j from 1 to 3: 2 1, 2 2, 2 3": "mis:for_loop->ForLoopPartSkipped"
 },
 "unexplained_clauses": [],
 "stop_reason": "sufficient",
 "depth_reached": 4,
 "llm_calls": 0,
 "llm_cache_hits": 72,
 "llm_calls_by_step": {
  "prune": 24,
  "label": 48
 }
}
```
